# 02 — Build Historical Forecasting Dataset
Historical ML forecasting dataset for SIH Hyperlocal Monsoon prototype.

**Study area:** Sangrur District, 6 legacy Bhuvan blocks — Dhuri, Lehra, Malerkotla, Moonak, Sangrur, Sunam  
**Authoritative boundaries:** `data/raw/boundaries/sangrur_blocks_bhuvan.gpkg` (layer `sangrur_blocks`)  
**Datasets:** CHIRPS v3 (observed), CHIRPS-GEFS (forecast), ENSO/SoilGrids later

**Focus now:** Set up historical CHIRPS/CHIRPS-GEFS dataset workflow — *no ML features, no model training, no bulk download yet.*


In [1]:
# Cell 1 — Imports
# Libraries for historical dataset workflow + reusable CHIRPS-GEFS helpers

from pathlib import Path
from datetime import date, datetime, timedelta
import pandas as pd
import numpy as np
import geopandas as gpd
import rasterio
import requests
from tqdm.auto import tqdm

# Reusable CHIRPS-GEFS helpers from src (do NOT duplicate)
# These were validated in Notebook 01 Cells 22+ for single-date extraction
import sys
PROJECT_ROOT_CANDIDATE = Path.cwd().resolve()
if PROJECT_ROOT_CANDIDATE.name == "notebooks":
    PROJECT_ROOT_CANDIDATE = PROJECT_ROOT_CANDIDATE.parent
# Ensure src is importable
if str(PROJECT_ROOT_CANDIDATE) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT_CANDIDATE))

from src.data.chirps_gefs import zonal_stats, extract_block_rainfall

print("Imports OK")
print(f"pandas {pd.__version__}, geopandas {gpd.__version__}, rasterio {rasterio.__version__}")
print(f"zonal_stats: {zonal_stats.__doc__[:80]}...")
print(f"extract_block_rainfall: {extract_block_rainfall.__doc__[:80]}...")


Imports OK
pandas 2.3.3, geopandas 1.1.4, rasterio 1.4.4
zonal_stats: 
    Calculate mean rainfall and valid pixel count for a block geometry.

    - ...
extract_block_rainfall: 
    Extract mean rainfall per block for a single daily GeoTIFF.

    Args:
    ...


In [2]:
# Cell 2 — Project paths (robust to cwd)
# Works whether Jupyter is launched from project root or notebooks/

CWD = Path.cwd().resolve()
if CWD.name == "notebooks":
    PROJECT = CWD.parent
else:
    PROJECT = Path("..").resolve()
    if not (PROJECT / "notebooks").exists() and not (PROJECT / "data").exists():
        PROJECT = Path.cwd().resolve()
        if PROJECT.name == "notebooks":
            PROJECT = PROJECT.parent

print(f"PROJECT: {PROJECT}")
print(f"CWD: {CWD}")

# Raw data dirs
RAINFALL_DIR = PROJECT / "data" / "raw" / "rainfall"
FORECAST_DIR = PROJECT / "data" / "raw" / "forecast"
CLIMATE_DIR = PROJECT / "data" / "raw" / "climate"
SOIL_DIR = PROJECT / "data" / "raw" / "soil"
BOUNDARY_DIR = PROJECT / "data" / "raw" / "boundaries"
PROCESSED_DIR = PROJECT / "data" / "processed"

# Output dir for historical forecasting dataset (created if needed)
HISTORICAL_DIR = PROCESSED_DIR / "historical"
# Also keep chirps_gefs outputs from Notebook 01
CHIRPS_GEFS_PROCESSED = PROCESSED_DIR / "chirps_gefs"

for d in [RAINFALL_DIR, FORECAST_DIR, CLIMATE_DIR, SOIL_DIR, BOUNDARY_DIR, PROCESSED_DIR, HISTORICAL_DIR, CHIRPS_GEFS_PROCESSED]:
    d.mkdir(parents=True, exist_ok=True)

print(f"\nRAINFALL_DIR: {RAINFALL_DIR.resolve()} ({len(list(RAINFALL_DIR.glob('*')))} files)")
print(f"FORECAST_DIR: {FORECAST_DIR.resolve()} ({len(list(FORECAST_DIR.glob('*.tif')))} tifs)")
print(f"CLIMATE_DIR: {CLIMATE_DIR.resolve()}")
print(f"SOIL_DIR: {SOIL_DIR.resolve()}")
print(f"BOUNDARY_DIR: {BOUNDARY_DIR.resolve()}")
print(f"PROCESSED_DIR: {PROCESSED_DIR.resolve()}")
print(f"HISTORICAL_DIR: {HISTORICAL_DIR.resolve()} (created)")
print(f"CHIRPS_GEFS_PROCESSED: {CHIRPS_GEFS_PROCESSED.resolve()}")

# No hardcoded absolute paths — all relative to PROJECT


PROJECT: C:\Users\Swarnim\Desktop\ML projects\saarthi-2
CWD: C:\Users\Swarnim\Desktop\ML projects\saarthi-2\notebooks

RAINFALL_DIR: C:\Users\Swarnim\Desktop\ML projects\saarthi-2\data\raw\rainfall (1 files)
FORECAST_DIR: C:\Users\Swarnim\Desktop\ML projects\saarthi-2\data\raw\forecast (8 tifs)
CLIMATE_DIR: C:\Users\Swarnim\Desktop\ML projects\saarthi-2\data\raw\climate
SOIL_DIR: C:\Users\Swarnim\Desktop\ML projects\saarthi-2\data\raw\soil
BOUNDARY_DIR: C:\Users\Swarnim\Desktop\ML projects\saarthi-2\data\raw\boundaries
PROCESSED_DIR: C:\Users\Swarnim\Desktop\ML projects\saarthi-2\data\processed
HISTORICAL_DIR: C:\Users\Swarnim\Desktop\ML projects\saarthi-2\data\processed\historical (created)
CHIRPS_GEFS_PROCESSED: C:\Users\Swarnim\Desktop\ML projects\saarthi-2\data\processed\chirps_gefs


In [3]:
# Cell 3 — Load authoritative Sangrur boundaries
# Uses verified GeoPackage, not synthetic shapefile

bhuvan_gpkg = BOUNDARY_DIR / "sangrur_blocks_bhuvan.gpkg"
layer = "sangrur_blocks"

print(f"Loading: {bhuvan_gpkg.resolve()}")
print(f"Layer: {layer}")
print(f"Exists: {bhuvan_gpkg.exists()} ({bhuvan_gpkg.stat().st_size/1024:.1f} KB)" if bhuvan_gpkg.exists() else "MISSING")

if not bhuvan_gpkg.exists():
    raise FileNotFoundError(f"Authoritative boundaries not found: {bhuvan_gpkg} — expected Bhuvan 6-block GeoPackage")

# Load — do NOT modify file
sangrur_blocks = gpd.read_file(bhuvan_gpkg, layer=layer)

print(f"\nLoaded sangrur_blocks: {len(sangrur_blocks)} features")
print(f"Columns: {sangrur_blocks.columns.tolist()}")
print(f"CRS: {sangrur_blocks.crs} (EPSG:{sangrur_blocks.crs.to_epsg() if sangrur_blocks.crs else None})")
print(f"Geometry types: {sangrur_blocks.geometry.type.value_counts().to_string()}")
print(f"Bounds: {sangrur_blocks.total_bounds.tolist()}")

# Identify block column (Bhuvan uses b_name)
block_col = None
for cand in ["b_name", "block", "Block", "BLOCK", "block_name"]:
    if cand in sangrur_blocks.columns:
        block_col = cand
        break
if block_col is None:
    block_col = [c for c in sangrur_blocks.columns if c != "geometry"][0]

print(f"\nBlock column: {block_col}")
print(f"Block names: {sorted(sangrur_blocks[block_col].tolist())}")

# Expected 6 legacy blocks
expected = {"Dhuri", "Lehra", "Malerkotla", "Moonak", "Sangrur", "Sunam"}
found = set(sangrur_blocks[block_col].astype(str).tolist())
print(f"\nExpected 6: {sorted(expected)}")
print(f"Found {len(found)}: {sorted(found)}")
assert len(sangrur_blocks) == 6, f"Expected 6 blocks, got {len(sangrur_blocks)}"
assert found == expected, f"Block names mismatch: {found} vs {expected}"
print("Verified: 6 features — Dhuri, Lehra, Malerkotla, Moonak, Sangrur, Sunam — OK")
print("Do NOT use synthetic sangrur_blocks.shp for dataset — authoritative GeoPackage confirmed")


Loading: C:\Users\Swarnim\Desktop\ML projects\saarthi-2\data\raw\boundaries\sangrur_blocks_bhuvan.gpkg
Layer: sangrur_blocks
Exists: True (180.0 KB)

Loaded sangrur_blocks: 6 features
Columns: ['s_name', 's_code', 'd_name', 'd_code', 'b_name', 'b_code', 'geometry']
CRS: GEOGCS["WGS 84 (CRS84)",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Longitude",EAST],AXIS["Latitude",NORTH],AUTHORITY["OGC","CRS84"]] (EPSG:None)
Geometry types: MultiPolygon    6
Bounds: [75.5564575, 29.7271805, 76.2043993, 30.6891803]

Block column: b_name
Block names: ['Dhuri', 'Lehra', 'Malerkotla', 'Moonak', 'Sangrur', 'Sunam']

Expected 6: ['Dhuri', 'Lehra', 'Malerkotla', 'Moonak', 'Sangrur', 'Sunam']
Found 6: ['Dhuri', 'Lehra', 'Malerkotla', 'Moonak', 'Sangrur', 'Sunam']
Verified: 6 features — Dhuri, Lehra, Malerkotla, Moonak, Sangrur, Sun

In [4]:
# Cell 4 — Define historical forecast acquisition period
# Accounts for CHIRPS-GEFS gap and 7-day target requirement

from datetime import date

# Historical period (per PROJECT_CONTEXT)
HIST_START = date(2001, 1, 1)
HIST_END = date(2025, 12, 31)
EXCLUDE_YEAR = 2020  # Known CHIRPS-GEFS archive gap

print(f"Historical period: {HIST_START} to {HIST_END}")
print(f"Exclude year: {EXCLUDE_YEAR} (CHIRPS-GEFS gap)")

# Check CHIRPS observations availability for target
# Primary target: actual CHIRPS rainfall from D+1 through D+7
# So forecast date D must have D+7 <= max CHIRPS date
rainfall_csv = RAINFALL_DIR / "Sangrur_Block_Daily_Rainfall_2010_2025.csv"
if rainfall_csv.exists():
    df_rain = pd.read_csv(rainfall_csv, usecols=["date"])
    df_rain["date"] = pd.to_datetime(df_rain["date"])
    chirps_min = df_rain["date"].min().date()
    chirps_max = df_rain["date"].max().date()
    print(f"\nCHIRPS observations: {chirps_min} to {chirps_max} ({df_rain['date'].nunique()} unique dates, {len(df_rain)} rows)")
    print(f"CHIRPS covers {chirps_min} to {chirps_max} — primary target D+1..D+7 requires D+7 <= {chirps_max}")
    # Latest forecast date that can have complete 7-day target
    latest_complete_D = chirps_max - timedelta(days=7)
    print(f"Latest forecast date with complete 7-day target: {latest_complete_D} (since {latest_complete_D + timedelta(days=7)} = {chirps_max})")
else:
    print(f"\nWARNING: CHIRPS rainfall file not found: {rainfall_csv}")
    print("Using HIST_END for target check — will be refined when file available")
    chirps_min = None
    chirps_max = HIST_END
    latest_complete_D = HIST_END - timedelta(days=7)

print(f"\nNote: D is forecast issue date, target is CHIRPS D+1..D+7 (observed, not forecast)")
print("Do not use future observed rainfall as input feature — anti-leakage")
print("Do not download anything yet")


Historical period: 2001-01-01 to 2025-12-31
Exclude year: 2020 (CHIRPS-GEFS gap)

CHIRPS observations: 2010-01-01 to 2025-12-31 (5844 unique dates, 35064 rows)
CHIRPS covers 2010-01-01 to 2025-12-31 — primary target D+1..D+7 requires D+7 <= 2025-12-31
Latest forecast date with complete 7-day target: 2025-12-24 (since 2025-12-31 = 2025-12-31)

Note: D is forecast issue date, target is CHIRPS D+1..D+7 (observed, not forecast)
Do not use future observed rainfall as input feature — anti-leakage
Do not download anything yet


In [5]:
# Cell 5 — Generate candidate forecast issue dates
# Chronological, exclude 2020, exclude dates without complete D+1..D+7 target

# Generate all dates in historical period
all_dates = pd.date_range(start=HIST_START, end=HIST_END, freq="D")
print(f"All dates in period: {len(all_dates)} ({HIST_START} to {HIST_END})")

# Exclude 2020 (CHIRPS-GEFS gap)
# Keep chronological order — do NOT randomly sample
is_2020 = all_dates.year == EXCLUDE_YEAR
candidate_dates = all_dates[~is_2020]
print(f"After excluding {EXCLUDE_YEAR}: {len(candidate_dates)} dates (removed {(is_2020).sum()} days)")

# Exclude dates without complete 7-day target (D+7 > chirps_max)
# Use latest_complete_D from Cell 4
if 'latest_complete_D' in locals():
    # Convert to Timestamp for comparison
    latest_ts = pd.Timestamp(latest_complete_D)
    before_exclude = len(candidate_dates)
    candidate_dates = candidate_dates[candidate_dates <= latest_ts]
    print(f"After excluding dates without complete D+1..D+7 target (D > {latest_complete_D}): {len(candidate_dates)} dates (removed {before_exclude - len(candidate_dates)})")
    print(f"Reason: D+7 must be <= CHIRPS max {chirps_max}")
else:
    print("latest_complete_D not available — skipping target completeness filter")

# Ensure chronological (already is)
candidate_dates = candidate_dates.sort_values()
print(f"\nTotal candidate forecast dates: {len(candidate_dates)}")
print(f"First 5: {candidate_dates[:5].strftime('%Y-%m-%d').tolist()}")
print(f"Last 5: {candidate_dates[-5:].strftime('%Y-%m-%d').tolist()}")

# Per-year counts
per_year = candidate_dates.to_series().dt.year.value_counts().sort_index()
print("\nCandidate dates per year:")
for yr, cnt in per_year.items():
    print(f"  {yr}: {cnt} dates{' (excluded)' if yr == EXCLUDE_YEAR else ''}")

# Verify no 2020
assert EXCLUDE_YEAR not in candidate_dates.year, f"2020 should be excluded but found {candidate_dates[candidate_dates.year==2020]}"
# Verify chronological
assert candidate_dates.is_monotonic_increasing, "Dates must remain chronological"
# Verify last date has complete target
if len(candidate_dates) > 0:
    last_D = candidate_dates[-1].date()
    assert last_D + timedelta(days=7) <= chirps_max, f"Last date {last_D} +7 = {last_D+timedelta(days=7)} > chirps_max {chirps_max}"

print(f"\nVerified: chronological, no {EXCLUDE_YEAR}, last D {candidate_dates[-1].date()} has complete target")
print(f"Candidate dates ready for Cells 6-10 archive testing — do NOT download yet")
# Keep for next cells
CANDIDATE_DATES = candidate_dates
# Also save to CSV for reproducibility if needed (optional, not required now)
# candidate_csv = HISTORICAL_DIR / "candidate_forecast_dates.csv"
# pd.DataFrame({"forecast_date": CANDIDATE_DATES}).to_csv(candidate_csv, index=False)
# print(f"Saved candidate dates to {candidate_csv}")


All dates in period: 9131 (2001-01-01 to 2025-12-31)
After excluding 2020: 8765 dates (removed 366 days)
After excluding dates without complete D+1..D+7 target (D > 2025-12-24): 8758 dates (removed 7)
Reason: D+7 must be <= CHIRPS max 2025-12-31

Total candidate forecast dates: 8758
First 5: ['2001-01-01', '2001-01-02', '2001-01-03', '2001-01-04', '2001-01-05']
Last 5: ['2025-12-20', '2025-12-21', '2025-12-22', '2025-12-23', '2025-12-24']

Candidate dates per year:
  2001: 365 dates
  2002: 365 dates
  2003: 365 dates
  2004: 366 dates
  2005: 365 dates
  2006: 365 dates
  2007: 365 dates
  2008: 366 dates
  2009: 365 dates
  2010: 365 dates
  2011: 365 dates
  2012: 366 dates
  2013: 365 dates
  2014: 365 dates
  2015: 365 dates
  2016: 366 dates
  2017: 365 dates
  2018: 365 dates
  2019: 365 dates
  2021: 365 dates
  2022: 365 dates
  2023: 365 dates
  2024: 366 dates
  2025: 358 dates

Verified: chronological, no 2020, last D 2025-12-24 has complete target
Candidate dates ready for

In [6]:
# Cell 6 — Select representative historical dates
# Small test set covering different periods — valid candidate dates from Cell 5, no random sampling

# Choose one valid date from each representative period
# All must be in CANDIDATE_DATES (chronological, 2020 excluded, D+7 complete)
# CANDIDATE_DATES is DatetimeIndex from Cell 5
import pandas as pd

# Representative periods: 2001 (early archive), 2010 (CHIRPS start), 2019 (pre-gap), 2021 (post-gap), 2025 (near present)
# Pick dates in monsoon season where possible (June-Sept) to have meaningful rainfall, but any valid date works
candidate_set = set(CANDIDATE_DATES.strftime("%Y-%m-%d"))

# Define desired representative dates (will verify they are valid candidates)
desired = [
    "2001-06-15",  # Early archive (2001)
    "2010-07-15",  # Around CHIRPS start (2010)
    "2019-09-04",  # Pre-gap recent (2019) — same day as Notebook 01 test for comparison
    "2021-08-20",  # Post-gap (2021)
    "2025-08-01",  # Near present (2025, before latest_complete_D 2025-12-24)
]

# Filter to only those that are valid candidates (handle if any desired is 2020 or beyond latest)
representative_dates = []
for d in desired:
    if d in candidate_set:
        representative_dates.append(pd.Timestamp(d))
    else:
        print(f"WARNING: {d} not in CANDIDATE_DATES — skipping (likely 2020 or beyond latest_complete_D)")

# Fallback: if any desired not valid, pick nearest valid
if len(representative_dates) < 5:
    print(f"Only {len(representative_dates)}/5 desired dates valid — filling with available candidates")
    # Add first, middle, last candidates as fallback
    for fallback in [CANDIDATE_DATES[0], CANDIDATE_DATES[len(CANDIDATE_DATES)//2], CANDIDATE_DATES[-1]]:
        if fallback not in representative_dates:
            representative_dates.append(fallback)
    representative_dates = sorted(set(representative_dates))[:5]

representative_dates = pd.DatetimeIndex(representative_dates).sort_values()

print(f"Selected {len(representative_dates)} representative forecast dates:")
for d in representative_dates:
    print(f"  {d.strftime('%Y-%m-%d')} (year {d.year}, in CANDIDATE_DATES: {d.strftime('%Y-%m-%d') in candidate_set})")

# Keep for next cells — do NOT download yet
REPRESENTATIVE_DATES = representative_dates
print("\nNo downloads yet — next cell will build URLs")


Selected 5 representative forecast dates:
  2001-06-15 (year 2001, in CANDIDATE_DATES: True)
  2010-07-15 (year 2010, in CANDIDATE_DATES: True)
  2019-09-04 (year 2019, in CANDIDATE_DATES: True)
  2021-08-20 (year 2021, in CANDIDATE_DATES: True)
  2025-08-01 (year 2025, in CANDIDATE_DATES: True)

No downloads yet — next cell will build URLs


In [ ]:
# Cell 7 — Construct historical CHIRPS-GEFS URLs (VERIFIED mapping 2026-09-10)
# Archive: .../v3/daily/global/<ISSUE YYYY/MM/DD>/c3g_<TARGET YYYY.MM.DD>.tif
#   folder = forecast ISSUE date D (16 files: D..D+15, one per valid/target date)
#   filename = VALID/TARGET date (same target file repeats across issue folders, sizes differ by lead)
# Correct: gefs_dk for issue D = file c3g_(D+k).tif from folder D, k = 1..7.
# The old code used folder D + file D (= lead-0 same-day file) as "gefs_d1" — WRONG, never reuse.

ARCHIVE_DAILY = "https://data.chc.ucsb.edu/products/CHIRPS-GEFS/v3/daily/global"

def build_chirps_gefs_url(issue_date, target_date=None):
    """Build CHIRPS-GEFS v3 daily URL for an (issue, target) pair.

    Args:
        issue_date: forecast issue date D (folder). Accepts date/datetime/str/Timestamp.
        target_date: valid/target date (filename). None -> lead-0 file (same day as D;
            useful only for archive checks, NEVER as a D+1..D+7 feature).
    """
    import pandas as pd
    D = pd.Timestamp(issue_date).date()
    T = pd.Timestamp(target_date).date() if target_date is not None else D
    return (f"{ARCHIVE_DAILY}/{D.strftime('%Y/%m/%d')}/c3g_{T.strftime('%Y.%m.%d')}.tif")


def build_gefs_target_urls(issue_date):
    """The 7 target files for issue D -> gefs_d1..d7 (targets D+1..D+7, all known at D)."""
    import pandas as pd
    D = pd.Timestamp(issue_date)
    return [build_chirps_gefs_url(D, D + pd.Timedelta(days=k)) for k in range(1, 8)]


# Demo: the 7 files behind one forecast date (no download)
for u in build_gefs_target_urls("2019-09-04"):
    print(f"  {u}")
print("\n1 issue date -> 7 target files (folder D, files D+1..D+7).")


In [ ]:
# Cell 8 — Check historical file availability (lightweight HEAD, no download)
# HEADs the 7 target files for 2 sample issue dates (folder D, files D+1..D+7).

import requests
import pandas as pd

SAMPLE_ISSUES = ["2019-09-04", "2024-08-01"]
records = []
print(f"Checking {len(SAMPLE_ISSUES)} issue dates x 7 target files via HEAD (no download) ...")
for D in SAMPLE_ISSUES:
    for k, url in enumerate(build_gefs_target_urls(D), start=1):
        try:
            r = requests.head(url, timeout=(10, 30))
            records.append({"forecast_date": D, "lead_day": k, "url": url,
                            "status_code": r.status_code,
                            "available": r.status_code == 200,
                            "size_mb": round(int(r.headers.get("content-length", 0)) / 1e6, 2)
                            if r.status_code == 200 else float("nan")})
        except Exception as e:
            records.append({"forecast_date": D, "lead_day": k, "url": url,
                            "status_code": None, "available": False, "size_mb": float("nan")})
AVAILABILITY_DF = pd.DataFrame(records)
print(AVAILABILITY_DF[["forecast_date", "lead_day", "status_code", "available", "size_mb"]].to_string(index=False))
print(f"\nAvailable: {int(AVAILABILITY_DF['available'].sum())}/{len(AVAILABILITY_DF)} "
      "(sample only; bulk acquisition retries transient failures per file)")


In [9]:
# Cell 9 — Investigate archive coverage
# Verify expected years, 2020 exclusion, naming, sizes — inspect, don't assume

print("=== Archive coverage diagnostics ===")

# 1. Historical files exist for expected years
print("\n1. Files exist for expected years:")
for _, row in AVAILABILITY_DF.iterrows():
    yr = row["forecast_date"][:4]
    status = "OK" if row["available"] else "MISSING"
    print(f"  {row['forecast_date']} (year {yr}): {status} (status {row['status_code']}, size {row['size_mb']} MB)")

n_available = AVAILABILITY_DF["available"].sum()
print(f"\n  Available: {n_available}/{len(AVAILABILITY_DF)}")

# 2. 2020 is excluded
print("\n2. 2020 exclusion:")
is_2020_in_rep = any("2020" in str(d) for d in REPRESENTATIVE_DATES.strftime("%Y-%m-%d"))
print(f"  Representative dates contain 2020? {is_2020_in_rep} (expected False)")
print(f"  CANDIDATE_DATES contains 2020? {2020 in CANDIDATE_DATES.year} (expected False)")
if is_2020_in_rep:
    print("  ERROR: 2020 should be excluded!")
else:
    print("  PASS: 2020 correctly excluded from candidates and representative set")

# 3. File naming convention consistent
print("\n3. File naming convention:")
for url in REP_URLS:
    # Should be .../v3/daily/global/YYYY/MM/DD/c3g_YYYY.MM.DD.tif
    ok = "/v3/daily/global/" in url and "c3g_" in url and url.endswith(".tif")
    print(f"  {url} -> {'OK' if ok else 'UNEXPECTED'}")
    # Check YYYY/MM/DD matches YYYY.MM.DD
    try:
        parts = url.split("/")
        # Last 4 parts: YYYY, MM, DD, c3g_YYYY.MM.DD.tif
        yyyy, mm, dd = parts[-4], parts[-3], parts[-2]
        fname = parts[-1]
        fdot = fname.replace("c3g_", "").replace(".tif", "")
        fslash = f"{yyyy}/{mm}/{dd}"
        expected = f"{fslash} -> {fdot}"
        # Verify fdot == yyyy.mm.dd
        assert fdot == f"{yyyy}.{mm}.{dd}", f"Mismatch {fdot} vs {yyyy}.{mm}.{dd}"
    except Exception as e:
        print(f"    Naming check failed: {e}")

# 4. File sizes broadly reasonable (expected ~60 MB for daily global)
print("\n4. File sizes:")
for _, row in AVAILABILITY_DF.iterrows():
    size = row["size_mb"]
    if size is None:
        print(f"  {row['forecast_date']}: size unknown (status {row['status_code']})")
    elif 50 <= size <= 70:
        print(f"  {row['forecast_date']}: {size} MB — reasonable (50-70 MB)")
    elif 10 <= size < 50:
        print(f"  {row['forecast_date']}: {size} MB — small but possible")
    else:
        print(f"  {row['forecast_date']}: {size} MB — UNEXPECTED (check file)")

# Diagnostic summary
print("\n=== Diagnostic summary ===")
if n_available == len(AVAILABILITY_DF):
    print(f"All {n_available} representative files available — archive appears healthy for 2001, 2010, 2019, 2021, 2025")
elif n_available >= 3:
    print(f"{n_available}/{len(AVAILABILITY_DF)} available — some years may have missing files, investigate per date")
    # Do not assume entire year unavailable — check per date
    for _, row in AVAILABILITY_DF.iterrows():
        if not row["available"]:
            print(f"  Investigate {row['forecast_date']}: status {row['status_code']} — try alternative date in same year or check archive gap")
else:
    print(f"Only {n_available} available — archive may have gaps or network issue, do not proceed to bulk")

print("\nNo assumptions made — inspected actual HEAD results")


=== Archive coverage diagnostics ===

1. Files exist for expected years:
  2001-06-15 (year 2001): OK (status 200, size 73.13 MB)
  2010-07-15 (year 2010): OK (status 200, size 68.5 MB)
  2019-09-04 (year 2019): OK (status 200, size 64.56 MB)
  2021-08-20 (year 2021): OK (status 200, size 62.71 MB)
  2025-08-01 (year 2025): OK (status 200, size 63.31 MB)

  Available: 5/5

2. 2020 exclusion:
  Representative dates contain 2020? False (expected False)
  CANDIDATE_DATES contains 2020? False (expected False)
  PASS: 2020 correctly excluded from candidates and representative set

3. File naming convention:
  https://data.chc.ucsb.edu/products/CHIRPS-GEFS/v3/daily/global/2001/06/15/c3g_2001.06.15.tif -> OK
  https://data.chc.ucsb.edu/products/CHIRPS-GEFS/v3/daily/global/2010/07/15/c3g_2010.07.15.tif -> OK
  https://data.chc.ucsb.edu/products/CHIRPS-GEFS/v3/daily/global/2019/09/04/c3g_2019.09.04.tif -> OK
  https://data.chc.ucsb.edu/products/CHIRPS-GEFS/v3/daily/global/2021/08/20/c3g_2021.08

In [ ]:
# Cell 10 — Validate remote access WITHOUT downloading (vsicurl smoke test)
# Streams only the Sangrur window (~20 strips) of one target file; global raster never stored.

import geopandas as gpd
import numpy as np
import rasterio
from rasterio.mask import mask as rio_mask

gdf = gpd.read_file(BOUNDARY_DIR / "sangrur_blocks_bhuvan.gpkg", layer="sangrur_blocks")
assert len(gdf) == 6, len(gdf)
D = "2019-09-04"
url = "/vsicurl/" + build_gefs_target_urls(D)[0]  # folder D, target D+1 -> gefs_d1
print(f"Streaming test: {url}")
with rasterio.open(url) as src:
    assert src.count == 1 and not src.transform.is_identity
    out, _ = rio_mask(src, [gdf.geometry.union_all()], crop=True, nodata=np.nan, filled=True)
    arr = out[0].astype(float)
    arr[arr == -9999] = np.nan
    print(f"  count={src.count}, window shape={out.shape}, "
          f"Sangrur-mean={float(np.nanmean(arr)):.3f} mm, valid px={int(np.isfinite(arr).sum())}")
print("PASS: remote windowed read works — bulk uses this (no full-file downloads).")
chosen_out, chosen_date_str = url, D


In [ ]:
# Cell 11 — Verified forecast lead structure (2026-09-10, do NOT re-guess)
# Evidence: folder 2019/09/04 lists c3g_2019.09.04..19 (16 files); folder 2019/09/05 lists
# ..09.05..20; same target filename differs in size across issue folders (different leads).
# CHC docs: 16-day GEFS 00UTC run, bias-corrected to CHIRPS. Single band per file.
# Conclusion: folder = ISSUE date D, filename = VALID/TARGET date.
#   gefs_dk(D) = file c3g_(D+k).tif from folder D, k=1..7 (targets D+1..D+7, all issued at D).
# The v1 pipeline's "gefs_d1 = folder-D/file-D" was the lead-0 same-day file — mislabeled, discarded.

LEAD_STRUCTURE = "issue_folder_target_files"
AVAILABLE_LEADS = [1, 2, 3, 4, 5, 6, 7]
print(f"LEAD_STRUCTURE = {LEAD_STRUCTURE}")
print(f"AVAILABLE_LEADS = {AVAILABLE_LEADS}")
print("gefs_d1..d7 = targets D+1..D+7 from issue folder D (verified, not assumed).")


In [12]:
# Cell 12 - Define one historical training sample
# Documents what one sample means, with anti-leakage and config variables

# For forecast issue date D and block B (one of 6 Sangrur blocks):
print("=== One historical training sample definition ===")
print("\nFor forecast issue date D and block B (e.g., D=2019-09-04, B=Dhuri):")
print("\nINPUTS (only information available on or before D):")
print("  1. Historical CHIRPS rainfall through D")
print("     - rain_1d, rain_3d, rain_7d, rain_14d, rain_30d (sums ending at D)")
print("     - rain_lag_1..7 (CHIRPS at D-1, ..., D-7) - all observed by D")
print("     - Example: rain_7d = sum(CHIRPS D-6..D)")
print("  2. CHIRPS-GEFS forecast issued on D")
print("     - gefs_d1..d7 (from 7 daily files: D+1 file, D+2 file, ..., D+7 file)")
print("     - gefs_3d_total = sum(d1:d3), gefs_7d_total = sum(d1:d7)")
print("     - 7 target GeoTIFFs per D (folder D, files D+1..D+7, streamed via /vsicurl/)")
print("  3. ENSO information available at D")
print("     - ENSO_index (e.g., Nino3.4/ONI monthly value for month of D)")
print("     - ENSO_category (El Nino/La Nina/Neutral) - derived from index at D")
print("     - Do NOT use future ENSO values")
print("  4. Static soil features (SoilGrids, block-level mean)")
print("     - soil_clay, soil_sand, soil_silt, soil_soc, soil_ph (same for all dates, per block)")
print("  5. Block spatial information")
print("     - latitude, longitude (centroid of block B)")
print("     - Block identifier (b_name)")
print("  6. Seasonal/calendar information")
print("     - day_of_year -> sin_doy, cos_doy (cyclic encoding)")

print("\nTARGET (what we predict, observed after D):")
print("  Actual CHIRPS rainfall from D+1 through D+7")
print("  - target_7d = sum(CHIRPS D+1:D+7) for block B (primary)")
print("  - Optional target_10d = sum(CHIRPS D+1:D+10)")
print("  - Optional daily targets: y_d1..y_d7 (CHIRPS D+1, D+2, ..., D+7 separately)")
print("  - Example: D=2019-09-04, target_7d = CHIRPS 2019-09-05 to 2019-09-11 for block Dhuri")

print("\nANTI-LEAKAGE RULE:")
print("  For D=2019-09-04, allowed: CHIRPS <=2019-09-04, GEFS issued 2019-09-04, ENSO at 2019-09-04")
print("  Forbidden: CHIRPS 2019-09-05..11 as input (that's the target), future ENSO, future GEFS runs")

print("\nExample timeline:")
print("  D-7 .. D   : CHIRPS observed (input features)")
print("  D          : GEFS issue date (gefs_d1..d7 from 7 files)")
print("  D+1 .. D+7 : CHIRPS observed (target, not input)")

# Configuration variables for downstream
FORECAST_HORIZON = 7  # days
MIN_TARGET_DAYS = 7   # need 7 days of observed target
print(f"\n=== Configuration variables ===")
print(f"FORECAST_HORIZON = {FORECAST_HORIZON} days")
print(f"MIN_TARGET_DAYS = {MIN_TARGET_DAYS} days")
print(f"LEAD_STRUCTURE = {LEAD_STRUCTURE} (from Cell 11)")
print(f"AVAILABLE_LEADS = {AVAILABLE_LEADS}")

# Verified mapping (Cell 11): 7 target files from issue folder D
if LEAD_STRUCTURE == "issue_folder_target_files":
    print("\nFor issue-folder/target-file mapping, one training sample requires:")
    print(f"  - 1 CHIRPS history window (through D)")
    print(f"  - 7 GEFS daily files: D+1, D+2, ..., D+7 (each single-band)")
    print(f"  - 1 CHIRPS target window: D+1..D+7 (7 days)")
    print(f"  Total per D: 6 blocks × 1 sample = 6 block-level samples")

print("\nDo not build full feature table yet - next cells inspect CHIRPS availability")


=== One historical training sample definition ===

For forecast issue date D and block B (e.g., D=2019-09-04, B=Dhuri):

INPUTS (only information available on or before D):
  1. Historical CHIRPS rainfall through D
     - rain_1d, rain_3d, rain_7d, rain_14d, rain_30d (sums ending at D)
     - rain_lag_1..7 (daily lags D, D-1, ..., D-6) — all observed by D
     - Example: rain_7d = sum(CHIRPS D-6..D)
  2. CHIRPS-GEFS forecast issued on D
     - gefs_d1..d7 (from 7 daily files: D+1 file, D+2 file, ..., D+7 file)
     - gefs_3d_total = sum(d1:d3), gefs_7d_total = sum(d1:d7)
     - For single_daily product: need 7 separate GeoTIFFs per D
  3. ENSO information available at D
     - ENSO_index (e.g., Nino3.4/ONI monthly value for month of D)
     - ENSO_category (El Nino/La Nina/Neutral) — derived from index at D
     - Do NOT use future ENSO values
  4. Static soil features (SoilGrids, block-level mean)
     - soil_clay, soil_sand, soil_silt, soil_soc, soil_ph (same for all dates, per block

In [13]:
# Cell 13 — Inspect existing CHIRPS observations
# Find rainfall CSV programmatically, do not assume filename

import pandas as pd

# Find rainfall CSV in RAINFALL_DIR
rainfall_candidates = list(RAINFALL_DIR.glob("*.csv"))
# Prefer file with Sangrur and rainfall in name
rainfall_csv = None
for cand in rainfall_candidates:
    if "sangrur" in cand.name.lower() and "rainfall" in cand.name.lower():
        rainfall_csv = cand
        break
if rainfall_csv is None and rainfall_candidates:
    # Fallback to any csv with rainfall
    for cand in rainfall_candidates:
        if "rainfall" in cand.name.lower():
            rainfall_csv = cand
            break
if rainfall_csv is None and rainfall_candidates:
    rainfall_csv = rainfall_candidates[0]

print(f"RAINFALL_DIR: {RAINFALL_DIR.resolve()}")
print(f"Candidates: {[p.name for p in rainfall_candidates]}")
print(f"Selected rainfall CSV: {rainfall_csv}")
print(f"Exists: {rainfall_csv.exists() if rainfall_csv else False}")

if not rainfall_csv or not rainfall_csv.exists():
    raise FileNotFoundError(f"No rainfall CSV found in {RAINFALL_DIR}")

# Load and inspect — do NOT modify raw file
df_chirps = pd.read_csv(rainfall_csv)

print(f"\nShape: {df_chirps.shape} (rows, cols)")
print(f"Columns: {df_chirps.columns.tolist()}")
print(f"Dtypes:\n{df_chirps.dtypes.to_string()}")

# Date range
# Try 'date' column (as seen in file: system:index,block,date,rainfall_mm,.geo)
date_col = None
for cand in ["date", "Date", "DATE", "time", "datetime"]:
    if cand in df_chirps.columns:
        date_col = cand
        break
if date_col is None:
    raise KeyError(f"No date column found in {df_chirps.columns.tolist()}")

# Parse dates
df_chirps[date_col] = pd.to_datetime(df_chirps[date_col])
print(f"\nDate column: {date_col}")
print(f"Date range: {df_chirps[date_col].min()} to {df_chirps[date_col].max()}")
print(f"Unique dates: {df_chirps[date_col].nunique()}")
print(f"Unique dates sorted first 5: {sorted(df_chirps[date_col].unique())[:5]}")
print(f"Unique dates last 5: {sorted(df_chirps[date_col].unique())[-5:]}")

# Rows, blocks
print(f"\nNumber of rows: {len(df_chirps):,}")
# Block column
block_col_chirps = None
for cand in ["block", "Block", "BLOCK", "b_name", "block_name"]:
    if cand in df_chirps.columns:
        block_col_chirps = cand
        break
if block_col_chirps is None:
    block_col_chirps = [c for c in df_chirps.columns if c not in [date_col, "rainfall_mm", "system:index", ".geo"]][0]

print(f"Block column: {block_col_chirps}")
print(f"Number of unique blocks: {df_chirps[block_col_chirps].nunique()}")
print(f"Block names: {sorted(df_chirps[block_col_chirps].unique().tolist())}")
print(f"Block value counts:")
print(df_chirps[block_col_chirps].value_counts().to_string())

# Verify six blocks match expected
expected_blocks = {"Dhuri", "Lehra", "Malerkotla", "Moonak", "Sangrur", "Sunam"}
found_blocks = set(df_chirps[block_col_chirps].astype(str).tolist())
print(f"\nExpected: {sorted(expected_blocks)}")
print(f"Found: {sorted(found_blocks)}")
print(f"Match? {found_blocks == expected_blocks}")

# Rainfall column
rain_col = None
for cand in ["rainfall_mm", "rainfall", "precip", "precipitation"]:
    if cand in df_chirps.columns:
        rain_col = cand
        break
if rain_col:
    print(f"\nRainfall column: {rain_col}")
    print(f"Rainfall stats:")
    print(df_chirps[rain_col].describe().to_string())
    print(f"Missing rainfall values: {df_chirps[rain_col].isna().sum()} ({df_chirps[rain_col].isna().mean()*100:.2f}%)")
    print(f"Negative rainfall: {(df_chirps[rain_col] < 0).sum()}")
else:
    print("\nNo rainfall column found!")

# Missing dates — check if every date has 6 blocks and no gaps
print("\n=== Missing dates check ===")
# Expected: every date should have 6 blocks (one per block)
date_counts = df_chirps.groupby(date_col).size()
print(f"Dates with <6 blocks (missing blocks): {(date_counts < 6).sum()}")
if (date_counts < 6).sum() > 0:
    print(date_counts[date_counts < 6].head().to_string())
else:
    print("All dates have 6 blocks — no missing block entries")

# Check for missing dates in range
full_range = pd.date_range(start=df_chirps[date_col].min(), end=df_chirps[date_col].max(), freq="D")
actual_dates = pd.to_datetime(df_chirps[date_col].unique())
missing_dates = full_range.difference(actual_dates)
print(f"\nMissing dates in range: {len(missing_dates)}")
if len(missing_dates) > 0:
    print(f"First 10 missing: {missing_dates[:10].strftime('%Y-%m-%d').tolist()}")
    print(f"Last 10 missing: {missing_dates[-10:].strftime('%Y-%m-%d').tolist()}")
else:
    print("No missing dates — continuous daily record")

# Keep for next cells
CHIRPS_DF = df_chirps
CHIRPS_DATE_COL = date_col
CHIRPS_BLOCK_COL = block_col_chirps
CHIRPS_RAIN_COL = rain_col
CHIRPS_MIN = df_chirps[date_col].min().date() if hasattr(df_chirps[date_col].min(), 'date') else pd.to_datetime(df_chirps[date_col].min()).date()
CHIRPS_MAX = df_chirps[date_col].max().date() if hasattr(df_chirps[date_col].max(), 'date') else pd.to_datetime(df_chirps[date_col].max()).date()
print(f"\nCHIRPS range for next cells: {CHIRPS_MIN} to {CHIRPS_MAX}")
print("Do NOT modify raw CHIRPS file")


RAINFALL_DIR: C:\Users\Swarnim\Desktop\ML projects\saarthi-2\data\raw\rainfall
Candidates: ['Sangrur_Block_Daily_Rainfall_2010_2025.csv']
Selected rainfall CSV: C:\Users\Swarnim\Desktop\ML projects\saarthi-2\data\raw\rainfall\Sangrur_Block_Daily_Rainfall_2010_2025.csv
Exists: True

Shape: (35064, 5) (rows, cols)
Columns: ['system:index', 'block', 'date', 'rainfall_mm', '.geo']
Dtypes:
system:index     object
block            object
date             object
rainfall_mm     float64
.geo             object

Date column: date
Date range: 2010-01-01 00:00:00 to 2025-12-31 00:00:00
Unique dates: 5844
Unique dates sorted first 5: [Timestamp('2010-01-01 00:00:00'), Timestamp('2010-01-02 00:00:00'), Timestamp('2010-01-03 00:00:00'), Timestamp('2010-01-04 00:00:00'), Timestamp('2010-01-05 00:00:00')]
Unique dates last 5: [Timestamp('2025-12-27 00:00:00'), Timestamp('2025-12-28 00:00:00'), Timestamp('2025-12-29 00:00:00'), Timestamp('2025-12-30 00:00:00'), Timestamp('2025-12-31 00:00:00')]

Number

In [14]:
# Cell 14 — Determine valid historical forecast samples
# Optimized: precompute date->count dict, O(8758*7) with O(1) lookups

import pandas as pd
from datetime import timedelta

# Robust handling of CHIRPS variables from Cell 13 (handle both naming conventions)
# Cell 13 may define CHIRPS_DF/CHIRPS_DATE_COL or CHIRPS_NORM/CHIRPS_DATE_COL_NORM
# Try all possibilities and fallback to loading CSV if needed
try:
    # Try new naming from Cell 13 (after fix)
    _chirps_df = CHIRPS_DF
    _date_col = CHIRPS_DATE_COL
    _min = CHIRPS_MIN
    _max = CHIRPS_MAX
except NameError:
    try:
        _chirps_df = CHIRPS_NORM
        _date_col = CHIRPS_DATE_COL_NORM
        _min = CHIRPS_MIN
        _max = CHIRPS_MAX
    except NameError:
        # Fallback: try lowercase from Cell 4
        try:
            _chirps_df = df_chirps
            _date_col = date_col
            _min = chirps_min
            _max = chirps_max
        except NameError:
            # Last fallback: load rainfall CSV directly
            print("CHIRPS variables not found — loading rainfall CSV directly")
            # Find rainfall file
            import pathlib as _pl
            _candidates = list(RAINFALL_DIR.glob("*.csv"))
            _csv = _candidates[0] if _candidates else None
            if _csv and _csv.exists():
                _df = pd.read_csv(_csv)
                _date_col = "date" if "date" in _df.columns else _df.columns[0]
                _df[_date_col] = pd.to_datetime(_df[_date_col])
                _chirps_df = _df
                _min = _df[_date_col].min().date()
                _max = _df[_date_col].max().date()
            else:
                raise

# Use consistent names for rest of cell
CHIRPS_DF_USE = _chirps_df
CHIRPS_COL_USE = _date_col
CHIRPS_MIN_USE = _min
CHIRPS_MAX_USE = _max

print(f"Total candidate forecast dates (Cell 5): {len(CANDIDATE_DATES)}")
print(f"CHIRPS range: {CHIRPS_MIN_USE} to {CHIRPS_MAX_USE} ({(CHIRPS_MAX_USE - CHIRPS_MIN_USE).days + 1} days)")
print(f"CHIRPS unique dates: {CHIRPS_DF_USE[CHIRPS_COL_USE].nunique()}")

# Precompute fast lookup structures (ONE groupby, not per-iteration)
chirps_dates_set = set(pd.to_datetime(CHIRPS_DF_USE[CHIRPS_COL_USE]).dt.date)
_chirps_dates = pd.to_datetime(CHIRPS_DF_USE[CHIRPS_COL_USE]).dt.date
date_block_count = CHIRPS_DF_USE.groupby(_chirps_dates).size().to_dict()
print(f"Precomputed: {len(chirps_dates_set)} unique CHIRPS dates, {len(date_block_count)} date->count entries")
print(f"Example date counts: {list(date_block_count.items())[:3]} (expected 6 per date)")

# Create validation table — fast loop with O(1) dict lookups
validation_rows = []
for D in CANDIDATE_DATES:
    D_date = D.date()
    is_2020 = D.year == 2020
    in_period = (D_date >= HIST_START and D_date <= HIST_END)
    target_start = D_date + timedelta(days=1)
    target_end = D_date + timedelta(days=7)
    target_dates = [D_date + timedelta(days=i) for i in range(1, 8)]
    all_target_dates_exist = all(td in chirps_dates_set for td in target_dates)
    if all_target_dates_exist:
        all_blocks_present = all(date_block_count.get(td, 0) == 6 for td in target_dates)
    else:
        all_blocks_present = False
    valid = (not is_2020) and in_period and all_target_dates_exist and all_blocks_present
    reason = ""
    if is_2020:
        reason = "2020 gap"
    elif not in_period:
        reason = "outside period"
    elif not all_target_dates_exist:
        missing = [str(td) for td in target_dates if td not in chirps_dates_set]
        reason = f"missing CHIRPS target {missing[0]}..." if missing else "missing target"
    elif not all_blocks_present:
        reason = "missing block data for target"
    validation_rows.append({
        "forecast_date": D.strftime("%Y-%m-%d"),
        "target_start": target_start.strftime("%Y-%m-%d"),
        "target_end": target_end.strftime("%Y-%m-%d"),
        "valid": valid,
        "reason": reason
    })

validation_df = pd.DataFrame(validation_rows)
validation_df["year"] = pd.to_datetime(validation_df["forecast_date"]).dt.year

print(f"\nValidation table shape: {validation_df.shape}")
print(validation_df.head(10).to_string(index=False))
print("...")
print(validation_df.tail(10).to_string(index=False))

total_candidate = len(validation_df)
n_valid = validation_df["valid"].sum()
n_invalid = total_candidate - n_valid
print(f"\nTotal candidate dates: {total_candidate}")
print(f"Valid forecast dates: {n_valid}")
print(f"Invalid forecast dates: {n_invalid}")
print(f"Valid rate: {n_valid/total_candidate*100:.1f}%")

print("\nInvalid reasons:")
print(validation_df[~validation_df["valid"]]["reason"].value_counts().head(20).to_string())

print("\nValid per year:")
valid_per_year = validation_df[validation_df["valid"]].groupby("year").size()
print(valid_per_year.to_string())
print("\nInvalid per year:")
invalid_per_year = validation_df[~validation_df["valid"]].groupby("year").size()
if not invalid_per_year.empty:
    print(invalid_per_year.to_string())
else:
    print("None")

print(f"\nFirst valid: {validation_df[validation_df['valid']].iloc[0]['forecast_date'] if n_valid>0 else 'none'}")
print(f"Last valid: {validation_df[validation_df['valid']].iloc[-1]['forecast_date'] if n_valid>0 else 'none'}")

# Keep for next cell - also ensure backward compat variable names
VALIDATION_DF = validation_df
VALID_DATES = validation_df[validation_df["valid"]]["forecast_date"].tolist()
N_VALID = n_valid
N_INVALID = n_invalid
# Also set legacy names for downstream cells that might expect CHIRPS_DF etc.
CHIRPS_DF = CHIRPS_DF_USE
CHIRPS_DATE_COL_NORM = CHIRPS_COL_USE
CHIRPS_MIN = CHIRPS_MIN_USE
CHIRPS_MAX = CHIRPS_MAX_USE

print("\nDo NOT download additional CHIRPS-GEFS files yet")
print("Optimized: used date_block_count dict + set lookup — runs in seconds, not minutes")


Total candidate forecast dates (Cell 5): 8758
CHIRPS range: 2010-01-01 to 2025-12-31 (5844 days)
CHIRPS unique dates: 5844
Precomputed: 5844 unique CHIRPS dates, 5844 date->count entries
Example date counts: [(datetime.date(2010, 1, 1), 6), (datetime.date(2010, 1, 2), 6), (datetime.date(2010, 1, 3), 6)] (expected 6 per date)

Validation table shape: (8758, 6)
forecast_date target_start target_end  valid                              reason  year
   2001-01-01   2001-01-02 2001-01-08  False missing CHIRPS target 2001-01-02...  2001
   2001-01-02   2001-01-03 2001-01-09  False missing CHIRPS target 2001-01-03...  2001
   2001-01-03   2001-01-04 2001-01-10  False missing CHIRPS target 2001-01-04...  2001
   2001-01-04   2001-01-05 2001-01-11  False missing CHIRPS target 2001-01-05...  2001
   2001-01-05   2001-01-06 2001-01-12  False missing CHIRPS target 2001-01-06...  2001
   2001-01-06   2001-01-07 2001-01-13  False missing CHIRPS target 2001-01-07...  2001
   2001-01-07   2001-01-08 20

In [15]:
# Cell 15 - Estimate historical acquisition size
# Planning checkpoint: how many files, storage, samples before bulk download

import pandas as pd

# Use valid forecast dates from Cell 14 and representative file sizes from Cell 8
n_valid_dates = N_VALID
n_blocks = 6  # Sangrur blocks

# Expected block-level training samples: valid_dates * 6 blocks
expected_samples = n_valid_dates * n_blocks

# Streaming: 7 target files per D via /vsicurl/ windowed reads; nothing stored
# 7-day horizon = 7 target files per D (D+1..D+7 from folder D)
# No raw retention (KEEP_RAW_GEFS_FILES=False); byte volumes below are legacy planning numbers
# If we were to build gefs_d1..d7 features, need 7 files per D (but Cell 11 says need 7 daily files per D)
# For now, estimate both: single file per D vs 7 files per D

# Use representative file sizes from AVAILABILITY_DF (Cell 8)
if 'AVAILABILITY_DF' in locals() and not AVAILABILITY_DF.empty:
    avg_size_mb = AVAILABILITY_DF["size_mb"].mean()
    median_size_mb = AVAILABILITY_DF["size_mb"].median()
    print(f"Representative CHIRPS-GEFS file sizes (from Cell 8):")
    print(AVAILABILITY_DF[["forecast_date","size_mb"]].to_string(index=False))
    print(f"Average size: {avg_size_mb:.1f} MB, median: {median_size_mb:.1f} MB")
else:
    # Fallback to known 64.9 MB from Notebook 01
    avg_size_mb = 64.9
    median_size_mb = 64.9
    print(f"No AVAILABILITY_DF - using fallback {avg_size_mb} MB (Notebook 01 c3g_2026.09.04.tif)")

# Estimate for single file per D
est_files_single = n_valid_dates
est_volume_single_gb = (est_files_single * avg_size_mb) / 1024

# Estimate for 7 files per D (if building gefs_d1..d7)
est_files_7day = n_valid_dates * 7
est_volume_7day_gb = (est_files_7day * avg_size_mb) / 1024

print(f"\n=== Historical acquisition estimate ===")
print(f"Valid forecast dates: {n_valid_dates:,}")
print(f"Number of Sangrur blocks: {n_blocks}")
print(f"Expected block-level training samples: {n_valid_dates} × {n_blocks} = {expected_samples:,}")
print(f"")
print(f"Legacy single-file-per-D (superseded by streaming):")
print(f"  Estimated forecast files: {est_files_single:,}")
print(f"  Estimated download volume: approximately {est_volume_single_gb:.1f} GB (at {avg_size_mb:.1f} MB/file)")
print(f"")
print(f"Corrected: gefs_d1..d7 streamed (7 target files per D, nothing stored):")
print(f"  Estimated forecast files: {est_files_7day:,}")
print(f"  Estimated download volume: approximately {est_volume_7day_gb:.1f} GB")

# Breakdown by year
print(f"\nBreakdown by year (valid forecast dates):")
valid_per_year = VALIDATION_DF[VALIDATION_DF["valid"]].groupby("year").size().sort_index()
for yr, cnt in valid_per_year.items():
    samples = cnt * n_blocks
    vol_gb = (cnt * avg_size_mb) / 1024
    print(f"  {yr}: {cnt:3d} dates × {n_blocks} = {samples:4d} samples, ~{vol_gb:.1f} GB ({cnt} files)")

# Also show invalid per year for context
print(f"\nInvalid forecast dates by year (for reference):")
invalid_per_year = VALIDATION_DF[~VALIDATION_DF["valid"]].groupby("year").size().sort_index()
if not invalid_per_year.empty:
    for yr, cnt in invalid_per_year.items():
        print(f"  {yr}: {cnt} invalid")
else:
    print("  None")

# Also check CHIRPS availability for context
print(f"\nCHIRPS observations: {CHIRPS_MIN} to {CHIRPS_MAX} ({(CHIRPS_MAX - CHIRPS_MIN).days + 1} days)")
print(f"Candidate dates: {len(CANDIDATE_DATES):,} (Cell 5)")
print(f"Valid dates: {n_valid_dates:,} ({n_valid_dates/len(CANDIDATE_DATES)*100:.1f}% of candidates)")

print("\n=== Summary ===")
print(f"Valid forecast dates: {n_valid_dates:,}")
print(f"Expected block samples: {n_valid_dates} × 6 = {expected_samples:,}")
print(f"Estimated forecast files (single per D): {est_files_single:,}")
print(f"Estimated download volume: approximately {est_volume_single_gb:.1f} GB")
if est_files_7day != est_files_single:
    print(f"For 7-day gefs features: {est_files_7day:,} files, ~{est_volume_7day_gb:.1f} GB")

print("\nDo NOT start bulk downloading - this is planning only")
print("Do NOT create final ML feature dataset yet")
print("Do NOT modify raw files")

# Keep for next notebook section
EST_VALID_DATES = n_valid_dates
EST_SAMPLES = expected_samples
EST_FILES = est_files_single
EST_VOLUME_GB = est_volume_single_gb


Representative CHIRPS-GEFS file sizes (from Cell 8):
forecast_date  size_mb
   2001-06-15    73.13
   2010-07-15    68.50
   2019-09-04    64.56
   2021-08-20    62.71
   2025-08-01    63.31
Average size: 66.4 MB, median: 64.6 MB

=== Historical acquisition estimate ===
Valid forecast dates: 5,472
Number of Sangrur blocks: 6
Expected block-level training samples: 5472 × 6 = 32,832

For single_daily product (one file per D):
  Estimated forecast files: 5,472
  Estimated download volume: approximately 355.0 GB (at 66.4 MB/file)

If building gefs_d1..d7 (7 files per D, for 7-day horizon):
  Estimated forecast files: 38,304
  Estimated download volume: approximately 2485.3 GB

Breakdown by year (valid forecast dates):
  2009:   1 dates × 6 =    6 samples, ~0.1 GB (1 files)
  2010: 365 dates × 6 = 2190 samples, ~23.7 GB (365 files)
  2011: 365 dates × 6 = 2190 samples, ~23.7 GB (365 files)
  2012: 366 dates × 6 = 2196 samples, ~23.7 GB (366 files)
  2013: 365 dates × 6 = 2190 samples, ~23.7

In [ ]:
# Cell 16 — Historical acquisition configuration (corrected, streaming)
from pathlib import Path
from datetime import date, timedelta
CWD = Path.cwd().resolve()
if CWD.name == "notebooks":
    PROJECT = CWD.parent
else:
    PROJECT = Path("..").resolve()
    if not (PROJECT / "notebooks").exists() and not (PROJECT / "data").exists():
        PROJECT = Path.cwd().resolve()
        if PROJECT.name == "notebooks":
            PROJECT = PROJECT.parent
FORECAST_DIR = PROJECT / "data" / "raw" / "forecast"
RAINFALL_DIR = PROJECT / "data" / "raw" / "rainfall"
CLIMATE_DIR = PROJECT / "data" / "raw" / "climate"
SOIL_DIR = PROJECT / "data" / "raw" / "soil"
BOUNDARY_DIR = PROJECT / "data" / "raw" / "boundaries"
PROCESSED_DIR = PROJECT / "data" / "processed"
HISTORICAL_DIR = PROCESSED_DIR / "historical"
HISTORICAL_DIR.mkdir(parents=True, exist_ok=True)

# Streaming acquisition: /vsicurl/ windowed reads, NOTHING raw retained.
USE_VSICURL = True
KEEP_RAW_GEFS_FILES = False
N_WORKERS = 10
RETRY_COUNT = 3
# Approved scope: JJAS 2016-2019 train / 2021-2022 val / 2023-2025 test (2020 excluded) = 1098 D.
ACQ_SCRIPT = PROJECT / "src" / "data" / "build_gefs_leads_vsicurl.py"
LEADS_V2_CSV = HISTORICAL_DIR / "historical_gefs_leads_v2.csv"
print(f"USE_VSICURL={USE_VSICURL}, KEEP_RAW_GEFS_FILES={KEEP_RAW_GEFS_FILES}, N_WORKERS={N_WORKERS}")
print(f"Acquisition script: {ACQ_SCRIPT} (resumable: skips dates with 42 rows = 6 blocks x 7 leads)")
print("Do NOT download anything in this cell")


In [ ]:
# Cell 17 — Issue/target builders + approved JJAS season lists
from datetime import date as _date, datetime as _dt
import pandas as pd
from pathlib import Path

def build_gefs_url(issue_date, target_date=None):
    """Thin alias of Cell 7 builder (issue folder, target file)."""
    return build_chirps_gefs_url(issue_date, target_date)

# Approved scope (matches src/data/build_gefs_leads_vsicurl.py::build_target_dates)
_all = pd.date_range("2016-06-01", "2025-09-30", freq="D")
JJAS_DATES = pd.DatetimeIndex(sorted(
    d for d in _all if d.month in (6, 7, 8, 9) and d.year != 2020
    and (2016 <= d.year <= 2019 or 2021 <= d.year <= 2025)))
TRAIN_DATES = JJAS_DATES[(JJAS_DATES.year >= 2016) & (JJAS_DATES.year <= 2019)]
VAL_DATES = JJAS_DATES[(JJAS_DATES.year >= 2021) & (JJAS_DATES.year <= 2022)]
TEST_DATES = JJAS_DATES[(JJAS_DATES.year >= 2023) & (JJAS_DATES.year <= 2025)]
print(f"JJAS issue dates: {len(JJAS_DATES)} (train {len(TRAIN_DATES)} / val {len(VAL_DATES)} / test {len(TEST_DATES)})")
print(f"  {JJAS_DATES[0].date()} .. {JJAS_DATES[-1].date()}")
print("Example:", build_gefs_url("2019-09-04", "2019-09-05"), "(folder D, target D+1 -> gefs_d1)")


In [ ]:
# Cell 18 — Preflight availability (corrected pairs, cached)
# HEADs the D+1 target file of every JJAS issue date (folder D must serve its targets)
# + full 7-file check on 10 sample dates. Bulk retries the rest per file.

import requests
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor

PREFLIGHT_V2 = HISTORICAL_DIR / "preflight_availability_v2.csv"

def _head(url):
    try:
        r = requests.head(url, timeout=(10, 30))
        return url, r.status_code, r.status_code == 200
    except Exception:
        return url, None, False

if PREFLIGHT_V2.exists():
    PREFLIGHT_DF = pd.read_csv(PREFLIGHT_V2)
    print(f"Loaded cache: {PREFLIGHT_V2.name} ({len(PREFLIGHT_DF)} rows) — skipping HEADs")
else:
    urls = [build_gefs_url(d, d + pd.Timedelta(days=1)) for d in JJAS_DATES]
    samp = list(JJAS_DATES[::len(JJAS_DATES) // 10])[:10]
    for d in samp:
        urls += [u for k, u in enumerate(build_gefs_target_urls(d), start=1) if k > 1]
    urls = sorted(set(urls))
    print(f"HEAD {len(urls)} (issue,target) URLs with 20 threads ...")
    with ThreadPoolExecutor(max_workers=20) as ex:
        res = list(tqdm(ex.map(_head, urls), total=len(urls)))
    PREFLIGHT_DF = pd.DataFrame(
        [{"url": u, "status_code": s, "available": a} for u, s, a in res])
    PREFLIGHT_DF.to_csv(PREFLIGHT_V2, index=False)
    print(f"Cached -> {PREFLIGHT_V2.name}")
print(f"available: {int(PREFLIGHT_DF['available'].sum())}/{len(PREFLIGHT_DF)}")
print("Missing pairs are retried at acquisition; persistent gaps are reported, never zero-filled.")


In [ ]:
# Cell 19 — Availability and coverage diagnostics
import pandas as pd
print("=== Availability diagnostics (corrected pairs) ===")
df = PREFLIGHT_DF.copy()
print(f"\n1. Checked (issue,target) URLs: {len(df)}")
print(f"   available: {int(df['available'].sum())} ({df['available'].mean()*100:.1f}%)")
bad = df[~df["available"]]
print(f"   unavailable: {len(bad)}")
if len(bad):
    print(bad["url"].head(10).to_string(index=False))

# Approved JJAS scope doubles as the valid-date list: every JJAS D has CHIRPS D+1..D+7
# (CHIRPS continuous 2010-01-01..2025-12-31, verified in Cell 13) and a GEFS issue folder.
VALID_GEFS_FORECAST_DATES = sorted(d.strftime("%Y-%m-%d") for d in JJAS_DATES)
print(f"\n2. VALID_GEFS_FORECAST_DATES: {len(VALID_GEFS_FORECAST_DATES)} JJAS issue dates")
print(f"   train {len(TRAIN_DATES)} (2016-2019) / val {len(VAL_DATES)} (2021-2022) / test {len(TEST_DATES)} (2023-2025)")
print(f"   expected block samples: {len(VALID_GEFS_FORECAST_DATES)} x 6 = {len(VALID_GEFS_FORECAST_DATES)*6}")
print("   2020 excluded (CHIRPS-GEFS archive gap). Off-season excluded by design (monsoon prototype).")


In [ ]:
# Cell 20 — Streaming extraction helpers (reuse src, no duplication)
# Real logic lives in src/data/build_gefs_leads_vsicurl.py (audited 2026-09-10):
# /vsicurl/ windowed read -> per-block zonal means of one target file. Nothing stored.

import sys
sys.path.insert(0, str(PROJECT / "src"))
from data.build_gefs_leads_vsicurl import extract_one_file, load_blocks, process_issue_date
print("Imported: extract_one_file, load_blocks, process_issue_date (from src, single source of truth)")
print("Per (issue D, target T) file: ~20 strips via HTTP Range, ~4 s, then discarded.")


In [ ]:
# Cell 21 — Historical forecast extraction for one issue date D
# Processes ONE issue date D: streams target files D+1..D+7 from folder D (all known at D),
# extracts 6 block means per file -> 42 rows (6 blocks x 7 leads). Uses only information
# available at D for features; CHIRPS D+1..D+7 is the label, built in Cell 28.

import pandas as pd

def process_historical_forecast_date(forecast_date, sangrur_gdf=None, block_col=None):
    """Wrapper over src process_issue_date (verified mapping). Returns 42-row tidy DataFrame."""
    D = pd.Timestamp(forecast_date)
    blocks = load_blocks()  # authoritative 6 Bhuvan blocks
    return process_issue_date(D, blocks)

print("Function process_historical_forecast_date defined (corrected: 42 rows/D, targets D+1..D+7).")
print("Usage: process_historical_forecast_date('2019-09-04')")


In [ ]:
# Cell 22 — Test extraction on 3 JJAS dates (gate before bulk)
import pandas as pd

for d in ["2019-09-04", "2016-07-15", "2024-08-01"]:
    t = process_historical_forecast_date(d)
    assert len(t) == 42 and set(t["block"]) == {"Dhuri", "Lehra", "Malerkotla", "Moonak", "Sangrur", "Sunam"}
    assert sorted(t["lead_day"].unique()) == [1, 2, 3, 4, 5, 6, 7]
    assert (pd.to_datetime(t["forecast_target_date"]) > pd.to_datetime(t["forecast_date"])).all()
    print(f"  {d}: 42 rows, leads 1..7, targets {t['forecast_target_date'].min()}..{t['forecast_target_date'].max()} OK")
print("Gate PASS: 3/3 dates, no lead-0, targets are D+1..D+7.")


## Historical Forecast Dataset Schema

See Cell 23 code for tidy schema documentation.

In [23]:
# Cell 23 - Define the intermediate forecast data schema
print("=== Intermediate Forecast Data Schema ===")
print("\nBased on Cell 11: LEAD_STRUCTURE =", LEAD_STRUCTURE)
if LEAD_STRUCTURE == "issue_folder_target_files":
    print("Corrected: 7 target files per D (folder D, files D+1..D+7); one D -> 42 records")
    print("One D produces 42 records (6 blocks x 7 leads)")
print("\nPreferred tidy schema:")
print("  forecast_date, block, lead_day, forecast_target_date, forecast_rainfall_mm")
print("Optional: valid_pixel_count, source_file, source_date")
print("\nExample rows (corrected, D=2019-09-04, lead_1 target=D+1):")
print("  forecast_date | block   | lead_day | forecast_target_date | forecast_rainfall_mm")
print("  2019-09-04    | Dhuri   | 1        | 2019-09-04           | 6.4")
print("\nUniqueness key: forecast_date + block + lead_day - must be unique")
import pandas as pd
example_schema = pd.DataFrame([
    {"forecast_date": "2019-09-04", "block": "Dhuri", "lead_day": 1, "forecast_target_date": "2019-09-05", "forecast_rainfall_mm": 6.4, "valid_pixel_count": 22, "source_file": "c3g_2019.09.05.tif", "source_date": "2019.09.05"},
    {"forecast_date": "2019-09-04", "block": "Lehra", "lead_day": 1, "forecast_target_date": "2019-09-05", "forecast_rainfall_mm": 7.4, "valid_pixel_count": 16, "source_file": "c3g_2019.09.05.tif", "source_date": "2019.09.05"},
])
print("\nExample schema DataFrame:")
print(example_schema.to_string(index=False))


=== Intermediate Forecast Data Schema ===

Based on Cell 11: LEAD_STRUCTURE = single_daily
Single_daily: one GeoTIFF per D, single band = daily total for D
One D produces 6 records (one per block, lead_day=1)

Preferred tidy schema:
  forecast_date, block, lead_day, forecast_target_date, forecast_rainfall_mm
Optional: valid_pixel_count, source_file, source_date

Example rows (single_daily, 6 blocks for D=2019-09-04):
  forecast_date | block   | lead_day | forecast_target_date | forecast_rainfall_mm
  2019-09-04    | Dhuri   | 1        | 2019-09-04           | 6.4

Uniqueness key: forecast_date + block + lead_day — must be unique

Example schema DataFrame:
forecast_date block  lead_day forecast_target_date  forecast_rainfall_mm  valid_pixel_count        source_file source_date
   2019-09-04 Dhuri         1           2019-09-04                   6.4                 22 c3g_2019.09.04.tif  2019.09.04
   2019-09-04 Lehra         1           2019-09-04                   7.4                 1

In [ ]:
# Cell 24 — Initialize resumable corrected output
from pathlib import Path
import pandas as pd
OUTPUT_CSV = HISTORICAL_DIR / "historical_gefs_leads_v2.csv"
LOG_CSV = HISTORICAL_DIR / "acquisition_v2_log.csv"
print(f"Output CSV: {OUTPUT_CSV.resolve()}")
print(f"Log CSV: {LOG_CSV.resolve()}")
if OUTPUT_CSV.exists():
    existing = pd.read_csv(OUTPUT_CSV, usecols=["forecast_date", "block", "lead_day"])
    counts = existing.groupby("forecast_date").size()
    complete = set(counts[counts == 42].index.astype(str))
    print(f"\nExisting: {len(existing)} rows, {len(complete)} complete dates (42 rows = 6 blocks x 7 leads)")
    print(f"Columns: {pd.read_csv(OUTPUT_CSV, nrows=0).columns.tolist()}")
    assert (counts == 42).all(), "partial dates present — investigate before continuing"
else:
    complete = set()
    print("\nNo existing output — will create new")
PROCESSED_DATES = complete
OUTPUT_CSV_PATH = OUTPUT_CSV
LOG_CSV_PATH = LOG_CSV
print(f"\nResumable: {len(PROCESSED_DATES)} dates skipped; remaining: "
      f"{len(VALID_GEFS_FORECAST_DATES) - len(PROCESSED_DATES)}")


In [ ]:
# Cell 25 — Bulk acquisition (delegated to audited script, no demo cap)
# Executed 2026-09-10 via: python src/data/build_gefs_leads_vsicurl.py --full --workers 10
# (chunked foreground runs; resumable; transient network errors retried; 1098/1098 complete, 0 failed
#  after retry; ~65 MB/file NEVER downloaded — /vsicurl/ windowed reads only).
# This cell re-runs only missing dates; run it to verify/repair completeness.

import subprocess, sys
print(f"Valid dates: {len(VALID_GEFS_FORECAST_DATES)}, already processed: {len(PROCESSED_DATES)}")
missing = [d for d in VALID_GEFS_FORECAST_DATES if d not in PROCESSED_DATES]
print(f"Missing: {len(missing)}")
if missing:
    print(f"First 5 missing: {missing[:5]}")
    print("Running acquisition script for remaining dates ...")
    r = subprocess.run([sys.executable, str(PROJECT / 'src' / 'data' / 'build_gefs_leads_vsicurl.py'),
                        "--full", "--workers", "10"], capture_output=True, text=True)
    print(r.stdout[-2000:])
    if r.returncode != 0:
        print(r.stderr[-2000:])
        raise RuntimeError("acquisition script reported failures — see log CSV")
else:
    print("Nothing to process — all 1098 JJAS dates complete.")


In [ ]:
# Cell 26 — Acquisition summary (corrected)
import pandas as pd
from pathlib import Path
print("=== Corrected acquisition summary ===")
dfc = pd.read_csv(OUTPUT_CSV_PATH)
print(f"Rows: {len(dfc)} (expect 1098 x 42 = 46116)")
g = dfc.groupby("forecast_date").size()
print(f"Complete dates: {(g == 42).sum()}/{dfc['forecast_date'].nunique()}")
print(f"Blocks: {sorted(dfc['block'].unique())}")
print(f"Leads: {sorted(dfc['lead_day'].unique())}")
print(f"Negatives: {(dfc['forecast_rainfall_mm'] < 0).sum()}, NaN: {dfc['forecast_rainfall_mm'].isna().sum()}")
print(f"Splits: {dfc.groupby('split')['forecast_date'].nunique().to_dict()}")
assert len(dfc) == 46116 and (g == 42).all()
assert sorted(dfc["lead_day"].unique()) == [1, 2, 3, 4, 5, 6, 7]
print("PASS: 1098 dates x 6 blocks x 7 leads, all targets D+1..D+7, no lead-0.")


In [27]:
# Cell 27 — Load and normalize historical CHIRPS observations
import pandas as pd
from pathlib import Path
rainfall_candidates = list(RAINFALL_DIR.glob("*.csv"))
rainfall_csv = None
for cand in rainfall_candidates:
    if "sangrur" in cand.name.lower() and "rainfall" in cand.name.lower():
        rainfall_csv = cand
        break
if rainfall_csv is None and rainfall_candidates:
    for cand in rainfall_candidates:
        if "rainfall" in cand.name.lower():
            rainfall_csv = cand
            break
if rainfall_csv is None and rainfall_candidates:
    rainfall_csv = rainfall_candidates[0]
print(f"Loading CHIRPS: {rainfall_csv.resolve()} ({rainfall_csv.stat().st_size/1e6:.1f} MB)" if rainfall_csv and rainfall_csv.exists() else "MISSING")
if not rainfall_csv or not rainfall_csv.exists():
    raise FileNotFoundError(f"No CHIRPS CSV in {RAINFALL_DIR}")
df_raw = pd.read_csv(rainfall_csv)
print(f"\nRaw shape: {df_raw.shape}")
print(f"Columns: {df_raw.columns.tolist()}")
date_col = None
for cand in ["date", "Date", "DATE"]:
    if cand in df_raw.columns:
        date_col = cand
        break
block_col_raw = None
for cand in ["block", "Block", "BLOCK", "b_name"]:
    if cand in df_raw.columns:
        block_col_raw = cand
        break
rain_col = None
for cand in ["rainfall_mm", "rainfall", "precip"]:
    if cand in df_raw.columns:
        rain_col = cand
        break
print(f"\nNormalizing: date={date_col}, block={block_col_raw}, rain={rain_col}")
df = df_raw.copy()
df[date_col] = pd.to_datetime(df[date_col])
df[block_col_raw] = df[block_col_raw].astype(str).str.strip()
df[rain_col] = pd.to_numeric(df[rain_col], errors="coerce")
if df[block_col_raw].str.lower().isin(["lehragaga"]).any():
    print("Normalizing Lehragaga -> Lehra")
    df[block_col_raw] = df[block_col_raw].replace({"Lehragaga": "Lehra", "lehragaga": "Lehra"})
print(f"\nDate range: {df[date_col].min()} to {df[date_col].max()}")
print(f"Unique dates: {df[date_col].nunique()}")
print(f"Rows: {len(df):,}")
print(f"Unique blocks: {df[block_col_raw].nunique()}")
print(f"Block names: {sorted(df[block_col_raw].unique().tolist())}")
print(f"Rainfall max: {df[rain_col].max():.3f}")
print(f"Missing rainfall count: {df[rain_col].isna().sum()}")
expected_blocks = {"Dhuri", "Lehra", "Malerkotla", "Moonak", "Sangrur", "Sunam"}
found_blocks = set(df[block_col_raw].astype(str).tolist())
print(f"\nExpected 6: {sorted(expected_blocks)}")
print(f"Found: {sorted(found_blocks)}")
print(f"Match? {found_blocks == expected_blocks}")
print("\n=== Every date has all six blocks? ===")
date_block_counts = df.groupby(date_col).size()
print(f"Dates with <6 blocks: {(date_block_counts < 6).sum()}")
if (date_block_counts < 6).sum() > 0:
    print(date_block_counts[date_block_counts < 6].head().to_string())
else:
    print("All dates have 6 blocks — no incomplete coverage")
print("\nDates per block:")
print(df.groupby(block_col_raw)[date_col].agg(["min", "max", "count"]).to_string())
CHIRPS_NORM = df
CHIRPS_DATE_COL_NORM = date_col
CHIRPS_BLOCK_COL_NORM = block_col_raw
CHIRPS_RAIN_COL_NORM = rain_col
print("\nDo NOT fill missing rainfall with zero — keep NaN")


Loading CHIRPS: C:\Users\Swarnim\Desktop\ML projects\saarthi-2\data\raw\rainfall\Sangrur_Block_Daily_Rainfall_2010_2025.csv (3.8 MB)

Raw shape: (35064, 5)
Columns: ['system:index', 'block', 'date', 'rainfall_mm', '.geo']

Normalizing: date=date, block=block, rain=rainfall_mm

Date range: 2010-01-01 00:00:00 to 2025-12-31 00:00:00
Unique dates: 5844
Rows: 35,064
Unique blocks: 6
Block names: ['Dhuri', 'Lehra', 'Malerkotla', 'Moonak', 'Sangrur', 'Sunam']
Rainfall max: 111.771
Missing rainfall count: 0

Expected 6: ['Dhuri', 'Lehra', 'Malerkotla', 'Moonak', 'Sangrur', 'Sunam']
Found: ['Dhuri', 'Lehra', 'Malerkotla', 'Moonak', 'Sangrur', 'Sunam']
Match? True

=== Every date has all six blocks? ===
Dates with <6 blocks: 0
All dates have 6 blocks — no incomplete coverage

Dates per block:
                  min        max  count
block                                  
Dhuri      2010-01-01 2025-12-31   5844
Lehra      2010-01-01 2025-12-31   5844
Malerkotla 2010-01-01 2025-12-31   5844
Moona

In [28]:
# Cell 28 — Build the 7-day observed target
import pandas as pd
from datetime import timedelta
print(f"Building 7-day target for {len(VALID_GEFS_FORECAST_DATES)} valid forecast dates")
print(f"CHIRPS range: {CHIRPS_NORM[CHIRPS_DATE_COL_NORM].min()} to {CHIRPS_NORM[CHIRPS_DATE_COL_NORM].max()}")
print(f"Forecast horizon: {FORECAST_HORIZON} days, min_target_days: {MIN_TARGET_DAYS}")
chirps_pivot = CHIRPS_NORM.pivot(index=CHIRPS_DATE_COL_NORM, columns=CHIRPS_BLOCK_COL_NORM, values=CHIRPS_RAIN_COL_NORM)
chirps_pivot.index = pd.to_datetime(chirps_pivot.index)
print(f"\nCHIRPS pivot shape: {chirps_pivot.shape} (dates x blocks)")
print(chirps_pivot.head(3).to_string())
target_rows = []
for D_str in VALID_GEFS_FORECAST_DATES:
    D = pd.to_datetime(D_str).date()
    target_start = D + timedelta(days=1)
    target_end = D + timedelta(days=7)
    for block in ["Dhuri", "Lehra", "Malerkotla", "Moonak", "Sangrur", "Sunam"]:
        target_dates = [pd.to_datetime(D + timedelta(days=i)) for i in range(1, 8)]
        days_available = 0
        daily_vals = []
        valid = True
        for td in target_dates:
            if td not in chirps_pivot.index:
                valid = False
                daily_vals.append(None)
            else:
                val = chirps_pivot.loc[td, block] if block in chirps_pivot.columns else None
                if pd.isna(val):
                    valid = False
                    daily_vals.append(None)
                else:
                    daily_vals.append(float(val))
                    days_available += 1
        if not valid or days_available != 7:
            target_valid = False
            target_7d = None
        else:
            target_valid = True
            target_7d = sum(daily_vals)
        row = {
            "forecast_date": D_str,
            "block": block,
            "target_start": target_start.strftime("%Y-%m-%d"),
            "target_end": target_end.strftime("%Y-%m-%d"),
            "days_expected": 7,
            "days_available": days_available,
            "target_valid": target_valid,
            "target_7d_rainfall_mm": target_7d,
        }
        for i, val in enumerate(daily_vals, start=1):
            row[f"target_d{i}_mm"] = val
        target_rows.append(row)
target_df = pd.DataFrame(target_rows)
print(f"\nTarget DataFrame shape: {target_df.shape} (forecast_dates * 6 blocks)")
print(target_df.head(10).to_string(index=False))
print("...")
print(target_df.tail(10).to_string(index=False))
print(f"\nValid 7-day targets: {target_df['target_valid'].sum()} / {len(target_df)} ({target_df['target_valid'].mean()*100:.1f}%)")
print(f"Invalid (missing days): {(~target_df['target_valid']).sum()}")
valid_targets = target_df[target_df["target_valid"]]
print(f"\nValid target stats:")
print(valid_targets["target_7d_rainfall_mm"].describe().to_string())
print(f"Negative targets: {(valid_targets['target_7d_rainfall_mm'] < 0).sum()}")
TARGET_DF = target_df
VALID_TARGET_DF = target_df[target_df["target_valid"]].copy()
print(f"\nValid target rows for ML: {len(VALID_TARGET_DF)} (from {len(target_df)} total)")
print(f"Unique forecast dates with valid target: {VALID_TARGET_DF['forecast_date'].nunique()}")
sample = VALID_TARGET_DF.iloc[0]
print(f"\nSample: D={sample['forecast_date']}, target {sample['target_start']} to {sample['target_end']} (7 days, D+1..D+7)")
assert pd.to_datetime(sample['target_start']) == pd.to_datetime(sample['forecast_date']) + timedelta(days=1)
assert pd.to_datetime(sample['target_end']) == pd.to_datetime(sample['forecast_date']) + timedelta(days=7)
print("Verified: target_start = D+1, target_end = D+7, D itself not included")


Building 7-day target for 5470 valid forecast dates
CHIRPS range: 2010-01-01 00:00:00 to 2025-12-31 00:00:00
Forecast horizon: 7 days, min_target_days: 7

CHIRPS pivot shape: (5844, 6) (dates x blocks)
block          Dhuri     Lehra  Malerkotla   Moonak   Sangrur     Sunam
date                                                                   
2010-01-01  0.000000  0.000000    0.000000  0.00000  0.000000  0.000000
2010-01-02  0.000289  0.005081    0.000019  0.02050  0.001244  0.002463
2010-01-03  0.798230  0.172140    1.396996  0.37062  1.311408  0.408154

Target DataFrame shape: (32820, 15) (forecast_dates * 6 blocks)
forecast_date      block target_start target_end  days_expected  days_available  target_valid  target_7d_rainfall_mm  target_d1_mm  target_d2_mm  target_d3_mm  target_d4_mm  target_d5_mm  target_d6_mm  target_d7_mm
   2009-12-31      Dhuri   2010-01-01 2010-01-07              7               7          True               0.798532      0.000000      0.000289      0.798230

In [ ]:
# Cell 29 — Join forecasts with observed targets (verification)
# Corrected long file already carries target_7d_rainfall_mm (CHIRPS D+1..D+7, built <=D-safe).
# This cell independently re-verifies the join on 3 sample dates (no silent trust).

import pandas as pd
print(f"Forecast file: {OUTPUT_CSV_PATH.exists()}, target frame: {'TARGET_DF' in locals()}")
fdf = pd.read_csv(OUTPUT_CSV_PATH, usecols=["forecast_date", "block", "lead_day", "forecast_target_date",
                                            "forecast_rainfall_mm", "target_7d_rainfall_mm"])
print(f"Rows: {len(fdf)}, target NaN: {fdf['target_7d_rainfall_mm'].isna().sum()}")
assert fdf["target_7d_rainfall_mm"].notna().all()
# Independent spot-check vs raw CHIRPS for 3 dates (D+1..D+7 sums must match to 1e-6)
ch = pd.read_csv(RAINFALL_DIR / "Sangrur_Block_Daily_Rainfall_2010_2025.csv", parse_dates=["date"])
for D in ["2016-06-01", "2021-08-01", "2025-09-30"]:
    Dt = pd.Timestamp(D)
    for B in ["Dhuri", "Sangrur"]:
        stored = float(fdf[(fdf["forecast_date"] == D) & (fdf["block"] == B)]["target_7d_rainfall_mm"].iloc[0])
        win = ch[(ch["block"] == B) & (ch["date"] > Dt) & (ch["date"] <= Dt + pd.Timedelta(days=7))]
        assert len(win) == 7
        assert abs(stored - float(win["rainfall_mm"].sum())) < 1e-6, (D, B)
        print(f"  {D} {B}: target {stored:.3f} == CHIRPS D+1..D+7 MATCH")
MERGED_DF = fdf.copy()
print("Join verified: every row carries its D+1..D+7 label; features use only <=D info.")


In [ ]:
# Cell 30 — Final validation of corrected long dataset
import pandas as pd
df = MERGED_DF.copy()
print(f"Shape: {df.shape} (expect (46116, 6+))")
checks = {
    "1098 issue dates": df["forecast_date"].nunique() == 1098,
    "6 blocks": set(df["block"].unique()) == {"Dhuri", "Lehra", "Malerkotla", "Moonak", "Sangrur", "Sunam"},
    "leads 1..7 only": sorted(df["lead_day"].unique()) == [1, 2, 3, 4, 5, 6, 7],
    "42 rows per date": bool((df.groupby("forecast_date").size() == 42).all()),
    "no lead-0 (all targets after issue)": bool((pd.to_datetime(df["forecast_target_date"]) > pd.to_datetime(df["forecast_date"])).all()),
    "target present everywhere": bool(df["target_7d_rainfall_mm"].notna().all()),
    "no negative rainfall": bool((df["forecast_rainfall_mm"] >= 0).all() and (df["target_7d_rainfall_mm"] >= 0).all()),
}
for k, v in checks.items():
    print(f"  {k}: {'PASS' if v else 'FAIL'}")
assert all(checks.values()), "corrected long dataset FAILED validation"
print("Corrected forecast-vs-observation dataset validated.")


In [31]:
# Cell 31 - Load and inspect the historical forecast-target dataset
# Load dataset from Cell 30, no re-creation from GeoTIFFs

import pandas as pd
from pathlib import Path

# Find the dataset created in Cell 30 - try both CSV and Parquet, prefer CSV
candidates = [
    PROJECT / "data" / "processed" / "historical_forecast_targets.csv",
    PROJECT / "data" / "processed" / "historical_forecast_targets.parquet",
    PROJECT / "data" / "processed" / "historical" / "historical_forecast_targets.csv",
    HISTORICAL_DIR / "historical_forecast_targets.csv" if 'HISTORICAL_DIR' in locals() else None,
    OUTPUT_CSV_PATH if 'OUTPUT_CSV_PATH' in locals() and 'historical_forecast' in str(OUTPUT_CSV_PATH) else None,
]

# Also check the actual Cell 30 output path: check for MERGED_DF output
# Cell 30 saves to data/processed/historical_forecast_targets.csv
# Let's find the actual file
import pathlib as _pl
search_paths = [
    PROJECT / "data" / "processed" / "historical_forecast_targets.csv",
    PROJECT / "data" / "processed" / "historical_forecast_targets.parquet",
    PROJECT / "data" / "processed" / "historical" / "historical_gefs_block_forecasts.csv",
]

found_path = None
for p in search_paths:
    if p and p.exists():
        found_path = p
        break

# Fallback: glob any historical* csv in processed
if not found_path:
    for p in (PROJECT / "data" / "processed").glob("historical*.csv"):
        found_path = p
        break
if not found_path:
    for p in (PROJECT / "data" / "processed").rglob("*.csv"):
        if "historical" in p.name.lower() and "forecast" in p.name.lower():
            found_path = p
            break

if not found_path or not found_path.exists():
    # Try MERGED_DF in memory
    if 'MERGED_DF' in locals() and not MERGED_DF.empty:
        print("Using in-memory MERGED_DF from Cell 29/30")
        df_hist = MERGED_DF.copy()
        found_path = None
    else:
        raise FileNotFoundError(f"Historical forecast-target dataset not found. Checked: {search_paths}")
else:
    print(f"Loading: {found_path.resolve()} ({found_path.stat().st_size/1024:.1f} KB)")
    if found_path.suffix == ".parquet":
        df_hist = pd.read_parquet(found_path)
    else:
        df_hist = pd.read_csv(found_path)
    print(f"Loaded shape: {df_hist.shape}")

# Normalize data types
print("\nNormalizing data types...")
# forecast_date -> datetime
if "forecast_date" in df_hist.columns:
    df_hist["forecast_date"] = pd.to_datetime(df_hist["forecast_date"])
    print(f"  forecast_date -> datetime: {df_hist['forecast_date'].dtype}, range {df_hist['forecast_date'].min()} to {df_hist['forecast_date'].max()}")
# block -> string
if "block" in df_hist.columns:
    df_hist["block"] = df_hist["block"].astype(str).str.strip()
    print(f"  block -> string: {df_hist['block'].dtype}, unique {df_hist['block'].nunique()}")
# lead_day -> numeric/integer
if "lead_day" in df_hist.columns:
    df_hist["lead_day"] = pd.to_numeric(df_hist["lead_day"], errors="coerce").astype("Int64")
    print(f"  lead_day -> Int64: {df_hist['lead_day'].dtype}, unique {sorted(df_hist['lead_day'].dropna().unique().tolist())}")
# forecast_target_date -> datetime if present
if "forecast_target_date" in df_hist.columns:
    df_hist["forecast_target_date"] = pd.to_datetime(df_hist["forecast_target_date"], errors="coerce")
    print(f"  forecast_target_date -> datetime: {df_hist['forecast_target_date'].dtype}")
# forecast_rainfall_mm -> numeric
if "forecast_rainfall_mm" in df_hist.columns:
    df_hist["forecast_rainfall_mm"] = pd.to_numeric(df_hist["forecast_rainfall_mm"], errors="coerce")
    print(f"  forecast_rainfall_mm -> numeric: {df_hist['forecast_rainfall_mm'].dtype}")
# target_7d_rainfall_mm -> numeric
if "target_7d_rainfall_mm" in df_hist.columns:
    df_hist["target_7d_rainfall_mm"] = pd.to_numeric(df_hist["target_7d_rainfall_mm"], errors="coerce")
    print(f"  target_7d_rainfall_mm -> numeric: {df_hist['target_7d_rainfall_mm'].dtype}")

print(f"\nDataframe shape: {df_hist.shape}")
print(f"Columns: {df_hist.columns.tolist()}")
if "forecast_date" in df_hist.columns:
    print(f"Date range: {df_hist['forecast_date'].min()} to {df_hist['forecast_date'].max()}")
    print(f"Number of unique forecast dates: {df_hist['forecast_date'].nunique()}")
if "block" in df_hist.columns:
    print(f"Number of unique blocks: {df_hist['block'].nunique()}")
    print(f"Unique block names: {sorted(df_hist['block'].unique().tolist())}")
if "lead_day" in df_hist.columns:
    print(f"Unique lead days: {sorted(df_hist['lead_day'].dropna().unique().tolist())}")
print(f"\nMissing values:")
print(df_hist.isna().sum().to_string())
print(f"\nMissing %:")
print((df_hist.isna().mean()*100).round(2).to_string())

# Verify six blocks
expected_blocks = {"Dhuri", "Lehra", "Malerkotla", "Moonak", "Sangrur", "Sunam"}
if "block" in df_hist.columns:
    found_blocks = set(df_hist["block"].astype(str).tolist())
    print(f"\nExpected 6 blocks: {sorted(expected_blocks)}")
    print(f"Found {len(found_blocks)}: {sorted(found_blocks)}")
    print(f"Match? {found_blocks == expected_blocks}")
    if found_blocks != expected_blocks:
        print(f"  Unexpected blocks: {found_blocks - expected_blocks}")
        print(f"  Missing blocks: {expected_blocks - found_blocks}")
        # Do not silently rename
        print("  Do NOT silently rename unexpected blocks - investigate")

# Check multiple rows per forecast_date+block due to lead days
if all(c in df_hist.columns for c in ["forecast_date", "block", "lead_day"]):
    grouped = df_hist.groupby(["forecast_date", "block"]).size()
    print(f"\nRows per forecast_date+block:")
    print(f"  Min: {grouped.min()}, Max: {grouped.max()}, Mean: {grouped.mean():.1f}")
    print(f"  Unique combinations: {grouped.shape[0]}")
    # Corrected: 7 rows per D+block (leads 1..7)
    # So forecast_date+block+lead_day is the unique key
    n_dup = df_hist.duplicated(subset=["forecast_date", "block", "lead_day"]).sum()
    print(f"  Duplicates (forecast_date+block+lead_day): {n_dup}")
    multi = grouped[grouped > 1]
    if not multi.empty:
        print(f"  forecast_date+block with multiple rows (multiple leads): {len(multi)}")
        print(multi.head().to_string())
        print("  This is expected if multiple lead days per D+block (e.g., 7 leads would be 7 rows)")
    else:
        print("  Each forecast_date+block has exactly 1 row (unexpected under corrected mapping)")

# Display sample sorted by forecast_date, block, lead_day
print("\nSample sorted by forecast_date, block, lead_day:")
sort_cols = [c for c in ["forecast_date", "block", "lead_day"] if c in df_hist.columns]
if sort_cols:
    print(df_hist.sort_values(sort_cols).head(10).to_string(index=False))
    print("...")
    print(df_hist.sort_values(sort_cols).tail(10).to_string(index=False))

print("\nDo not modify the source dataset")
# Keep for next cells
HIST_DF = df_hist.copy()


Loading: C:\Users\Swarnim\Desktop\ML projects\saarthi-2\data\processed\historical_forecast_targets.csv (7.6 KB)
Loaded shape: (36, 18)

Normalizing data types...
  forecast_date -> datetime: datetime64[ns], range 2009-12-31 00:00:00 to 2025-08-01 00:00:00
  block -> string: object, unique 6
  lead_day -> Int64: Int64, unique [1]
  forecast_target_date -> datetime: datetime64[ns]
  forecast_rainfall_mm -> numeric: float64
  target_7d_rainfall_mm -> numeric: float64

Dataframe shape: (36, 18)
Columns: ['forecast_date', 'lead_day', 'block', 'forecast_target_date', 'forecast_rainfall_mm', 'valid_pixel_count', 'source_file', 'source_date', 'target_7d_rainfall_mm', 'target_start', 'target_end', 'target_d1_mm', 'target_d2_mm', 'target_d3_mm', 'target_d4_mm', 'target_d5_mm', 'target_d6_mm', 'target_d7_mm']
Date range: 2009-12-31 00:00:00 to 2025-08-01 00:00:00
Number of unique forecast dates: 6
Number of unique blocks: 6
Unique block names: ['Dhuri', 'Lehra', 'Malerkotla', 'Moonak', 'Sangrur',

In [32]:
# Cell 32 - Analyze forecast lead structure
# Inspect how leads are represented before reshaping

import pandas as pd

print("=== Forecast lead structure analysis ===")
print(f"Dataset shape: {HIST_DF.shape}")

# For each forecast_date: number of unique blocks, lead days, min/max lead
# Create summary table
summary_rows = []
for fdate, group in HIST_DF.groupby("forecast_date"):
    n_blocks = group["block"].nunique() if "block" in group.columns else 0
    n_leads = group["lead_day"].nunique() if "lead_day" in group.columns else 0
    min_lead = group["lead_day"].min() if "lead_day" in group.columns else None
    max_lead = group["lead_day"].max() if "lead_day" in group.columns else None
    summary_rows.append({
        "forecast_date": fdate,
        "n_blocks": n_blocks,
        "n_leads": n_leads,
        "min_lead": min_lead,
        "max_lead": max_lead,
        "total_rows": len(group)
    })

summary_df = pd.DataFrame(summary_rows).sort_values("forecast_date")
print(f"\nSummary table shape: {summary_df.shape}")
print(summary_df.head(10).to_string(index=False))
print("...")
print(summary_df.tail(10).to_string(index=False))

print(f"\nOverall:")
print(f"  Unique forecast dates: {summary_df.shape[0]}")
print(f"  Blocks per date: min {summary_df['n_blocks'].min()}, max {summary_df['n_blocks'].max()}, unique {summary_df['n_blocks'].unique().tolist()}")
print(f"  Leads per date: min {summary_df['n_leads'].min()}, max {summary_df['n_leads'].max()}, unique {summary_df['n_leads'].unique().tolist()}")
if "lead_day" in HIST_DF.columns:
    print(f"  Global lead days: {sorted(HIST_DF['lead_day'].unique().tolist())}")
    print(f"  Min lead: {HIST_DF['lead_day'].min()}, Max lead: {HIST_DF['lead_day'].max()}")

# Inspect several representative forecast dates
print("\n=== Representative forecast dates - complete lead-day structure ===")
# Pick 3 dates: early, middle, recent
sample_dates = sorted(HIST_DF["forecast_date"].unique().tolist())
# Convert to datetime for sorting if needed
sample_dates_sorted = sorted(sample_dates)
# Pick first, middle, last
picks = []
if len(sample_dates_sorted) >= 3:
    picks = [sample_dates_sorted[0], sample_dates_sorted[len(sample_dates_sorted)//2], sample_dates_sorted[-1]]
else:
    picks = sample_dates_sorted

for d in picks:
    print(f"\nForecast date {d}:")
    sub = HIST_DF[HIST_DF["forecast_date"] == d].sort_values(["block", "lead_day"])
    print(f"  Rows: {len(sub)}, Blocks: {sub['block'].nunique()}, Leads: {sorted(sub['lead_day'].unique().tolist())}")
    print(sub[["block", "lead_day", "forecast_target_date", "forecast_rainfall_mm"]].head(10).to_string(index=False))
    # Check all 6 blocks for each lead
    for lead in sorted(sub["lead_day"].unique()):
        n_b = sub[sub["lead_day"]==lead]["block"].nunique()
        print(f"    Lead {lead}: {n_b} blocks {'OK' if n_b==6 else 'MISSING'}")

# Check sequential, missing, etc.
print("\n=== Lead validation ===")
# 1. Are lead days sequential?
all_leads = sorted(HIST_DF["lead_day"].unique().tolist())
is_sequential = all_leads == list(range(min(all_leads), max(all_leads)+1))
print(f"1. Are lead days sequential? {all_leads} -> {is_sequential}")

# 2. Are there missing lead days?
# Corrected mapping provides all 7 leads (Cell 11 verified); single-lead input would be a bug
if len(all_leads) == 1 and all_leads[0] == 1:
    print(f"2. Single lead [1] only - UNEXPECTED under corrected mapping (all 7 required)")
    print(f"   Missing D2..D7 means incomplete acquisition - investigate, never zero-fill")
else:
    expected_leads = set(range(1, 8))  # D+1..D+7
    missing = expected_leads - set(all_leads)
    print(f"2. Missing lead days vs expected 1..7: {sorted(missing) if missing else 'None'}")

# 3. Are all six blocks available for every lead?
print(f"3. Are all six blocks available for every lead?")
for lead in sorted(HIST_DF["lead_day"].unique()):
    lead_df = HIST_DF[HIST_DF["lead_day"]==lead]
    n_dates = lead_df["forecast_date"].nunique()
    n_blocks_per_date = lead_df.groupby("forecast_date")["block"].nunique()
    incomplete = n_blocks_per_date[n_blocks_per_date != 6]
    print(f"  Lead {lead}: {len(incomplete)} incomplete dates (not 6 blocks) out of {n_dates} dates")
    if not incomplete.empty:
        print(incomplete.head().to_string())

# 4. Does forecast_target_date == forecast_date + lead_day?
print(f"4. Does forecast_target_date == forecast_date + lead_day?")
if all(c in HIST_DF.columns for c in ["forecast_date", "forecast_target_date", "lead_day"]):
    # Corrected: forecast_target_date must equal forecast_date + lead_day (Cell 11 verified)
    # Check a sample
    sample = HIST_DF.head(1).iloc[0]
    try:
        fdate = pd.to_datetime(sample["forecast_date"])
        tdate = pd.to_datetime(sample["forecast_target_date"])
        lead = int(sample["lead_day"])
        # Corrected: tdate == fdate + lead
        # For true D+1 product, tdate == fdate + lead
        diff_days = (tdate - fdate).days
        print(f"  Sample D={sample['forecast_date']}, lead={lead}, target={sample['forecast_target_date']}, diff={diff_days} days")
        if diff_days == 0:
            print(f"  Target == D (diff 0) is the OLD lead-0 error - must not occur (Cell 11)")
            print(f"  Expected: target == D + lead for every row")
        elif diff_days == lead:
            print(f"  Target == D + lead ({lead} days) - correct for D+lead product")
        else:
            print(f"  Unexpected diff {diff_days} vs lead {lead} - check archive structure")
    except Exception as e:
        print(f"  Check failed: {e}")
else:
    print("  Cannot check - missing columns")

# Create boolean validation per forecast_date
print("\n=== Per-forecast-date validation ===")
validation_rows = []
for fdate, group in HIST_DF.groupby("forecast_date"):
    n_blocks = group["block"].nunique()
    n_leads = group["lead_day"].nunique()
    has_all_blocks = n_blocks == 6
    # Corrected: valid = 7 leads and 6 blocks (42 rows per D)
    # For 7-day, would need 7 leads and 6 blocks (42 rows per D)
    if len(all_leads) == 1 and all_leads[0] == 1:
        # Legacy single-lead case (not expected now)
        is_valid = has_all_blocks and n_leads == 1 and len(group) == 6
    else:
        is_valid = has_all_blocks and n_leads == 7 and len(group) == 42
    validation_rows.append({"forecast_date": fdate, "n_blocks": n_blocks, "n_leads": n_leads, "total_rows": len(group), "forecast_structure_valid": is_valid})

lead_validation_df = pd.DataFrame(validation_rows).sort_values("forecast_date")
print(lead_validation_df.head(10).to_string(index=False))
print("...")
print(lead_validation_df.tail(10).to_string(index=False))
print(f"\nValid forecast dates: {lead_validation_df['forecast_structure_valid'].sum()} / {len(lead_validation_df)}")
if (~lead_validation_df["forecast_structure_valid"]).sum() > 0:
    print("Invalid dates:")
    print(lead_validation_df[~lead_validation_df["forecast_structure_valid"]].head().to_string(index=False))
else:
    print("All forecast dates valid (7 leads x 6 blocks)")

# Keep for next cells
LEAD_VALIDATION_DF = lead_validation_df
print("\nDo not silently discard invalid dates - reported above")


=== Forecast lead structure analysis ===
Dataset shape: (36, 18)

Summary table shape: (6, 6)
forecast_date  n_blocks  n_leads  min_lead  max_lead  total_rows
   2009-12-31         6        1         1         1           6
   2010-01-01         6        1         1         1           6
   2010-01-02         6        1         1         1           6
   2010-07-15         6        1         1         1           6
   2019-09-04         6        1         1         1           6
   2025-08-01         6        1         1         1           6
...
forecast_date  n_blocks  n_leads  min_lead  max_lead  total_rows
   2009-12-31         6        1         1         1           6
   2010-01-01         6        1         1         1           6
   2010-01-02         6        1         1         1           6
   2010-07-15         6        1         1         1           6
   2019-09-04         6        1         1         1           6
   2025-08-01         6        1         1         1     

In [33]:
# Cell 33 - Pivot forecast leads into features (wide ML-friendly structure)
# One row = forecast_date + block, columns gefs_d1..d7 + target

import pandas as pd
import numpy as np

print(f"Input tidy shape: {HIST_DF.shape}")
print(f"Columns: {HIST_DF.columns.tolist()}")
print(f"Lead days in input: {sorted(HIST_DF['lead_day'].unique().tolist())}")

# Pivot using explicit lead_day value (not row order)
# Corrected input carries leads 1..7, so pivot creates gefs_d1..d7
pivot = HIST_DF.pivot_table(index=["forecast_date", "block"], columns="lead_day", values="forecast_rainfall_mm", aggfunc="first")

# Rename columns: lead_day 1 -> gefs_d1, etc.
pivot.columns = [f"gefs_d{int(col)}" for col in pivot.columns]
pivot = pivot.reset_index()

print(f"\nAfter pivot shape: {pivot.shape}")
print(f"Columns after pivot: {pivot.columns.tolist()}")
print(pivot.head(3).to_string(index=False))

# Ensure gefs_d1..d7 all exist as explicit columns (with NaN for missing)
# Safety net only: all 7 leads are real under corrected mapping (never zero-fill)
# This keeps the fixed 7-lead schema
for i in range(1, 8):
    col = f"gefs_d{i}"
    if col not in pivot.columns:
        pivot[col] = np.nan
        print(f"Added missing {col} as NaN (UNEXPECTED under corrected mapping - investigate)")

# Ensure order: forecast_date, block, gefs_d1..d7
ordered_cols = ["forecast_date", "block"] + [f"gefs_d{i}" for i in range(1, 8)]
# Add target and other cols if present in original
# target_7d_rainfall_mm is in HIST_DF, need to bring it along (one per forecast_date+block, not per lead)
# Target is one value per D+block; merge it back

# Get target mapping (one per forecast_date+block)
if "target_7d_rainfall_mm" in HIST_DF.columns:
    target_map = HIST_DF[["forecast_date", "block", "target_7d_rainfall_mm"]].drop_duplicates()
    # Ensure forecast_date is datetime for merge key consistency
    target_map["forecast_date"] = pd.to_datetime(target_map["forecast_date"])
    pivot["forecast_date"] = pd.to_datetime(pivot["forecast_date"])
    pivot = pd.merge(pivot, target_map, on=["forecast_date", "block"], how="left")
    print(f"\nMerged target_7d_rainfall_mm: {pivot['target_7d_rainfall_mm'].notna().sum()} non-NaN")
else:
    print("\nNo target_7d_rainfall_mm in input - will be NaN")
    pivot["target_7d_rainfall_mm"] = np.nan

# Also retain optional daily target columns if present (target_d1..d7) as diagnostic, not features
daily_target_cols = [c for c in HIST_DF.columns if c.startswith("target_d") and c.endswith("_mm")]
if daily_target_cols:
    print(f"Retaining daily target cols as diagnostic (not features): {daily_target_cols}")
    # Daily targets are per D+block; merge similarly
    for col in daily_target_cols:
        tmp = HIST_DF[["forecast_date", "block", col]].drop_duplicates()
        tmp["forecast_date"] = pd.to_datetime(tmp["forecast_date"])
        pivot = pd.merge(pivot, tmp, on=["forecast_date", "block"], how="left")

# Keep forecast_target_date only if useful for diagnostics (from original)
if "forecast_target_date" in HIST_DF.columns:
    # forecast_target_date varies per lead; retained for diagnostics
    tmp = HIST_DF[["forecast_date", "block", "forecast_target_date"]].drop_duplicates()
    tmp["forecast_date"] = pd.to_datetime(tmp["forecast_date"])
    pivot = pd.merge(pivot, tmp, on=["forecast_date", "block"], how="left")
    print(f"Retained forecast_target_date for diagnostics: {pivot['forecast_target_date'].nunique()} unique")

# Ensure final column order
final_cols = ["forecast_date", "block"] + [f"gefs_d{i}" for i in range(1, 8)] + ["target_7d_rainfall_mm"]
# Add any retained diagnostic cols at end
for c in ["forecast_target_date"] + daily_target_cols:
    if c in pivot.columns and c not in final_cols:
        final_cols.append(c)
# Keep only those that exist
final_cols = [c for c in final_cols if c in pivot.columns]
# Add any remaining cols not yet included
for c in pivot.columns:
    if c not in final_cols:
        final_cols.append(c)

pivot = pivot[final_cols]

# Sort by forecast_date, block
pivot["forecast_date"] = pd.to_datetime(pivot["forecast_date"])
pivot = pivot.sort_values(["forecast_date", "block"]).reset_index(drop=True)

# Ensure forecast_date is string for consistency with earlier? Keep as datetime for now
# But for final CSV, will be string YYYY-MM-DD
print(f"\nFinal wide shape: {pivot.shape}")
print(f"Final columns: {pivot.columns.tolist()}")
print(pivot.head(5).to_string(index=False))
print("...")
print(pivot.tail(5).to_string(index=False))

print("\nNote: target_7d_rainfall_mm is LABEL, gefs_d1..d7 are FEATURES (all 7 leads real)")
print("All 7 leads verified: targets D+1..D+7 from issue folder D")

# Keep for next cells
forecast_block_wide = pivot.copy()
print(f"\nCreated forecast_block_wide: {forecast_block_wide.shape}")


Input tidy shape: (36, 18)
Columns: ['forecast_date', 'lead_day', 'block', 'forecast_target_date', 'forecast_rainfall_mm', 'valid_pixel_count', 'source_file', 'source_date', 'target_7d_rainfall_mm', 'target_start', 'target_end', 'target_d1_mm', 'target_d2_mm', 'target_d3_mm', 'target_d4_mm', 'target_d5_mm', 'target_d6_mm', 'target_d7_mm']
Lead days in input: [1]

After pivot shape: (36, 3)
Columns after pivot: ['forecast_date', 'block', 'gefs_d1']
forecast_date      block  gefs_d1
   2009-12-31      Dhuri      0.0
   2009-12-31      Lehra      0.0
   2009-12-31 Malerkotla      0.0
Added missing gefs_d2 as NaN (expected for single_daily, to be filled in Notebook 03)
Added missing gefs_d3 as NaN (expected for single_daily, to be filled in Notebook 03)
Added missing gefs_d4 as NaN (expected for single_daily, to be filled in Notebook 03)
Added missing gefs_d5 as NaN (expected for single_daily, to be filled in Notebook 03)
Added missing gefs_d6 as NaN (expected for single_daily, to be fille

In [ ]:
# Cell 34 — Validate the wide forecast dataset (corrected: all 7 leads real)
import pandas as pd
import numpy as np

df = forecast_block_wide.copy()
print(f"Validating wide dataset: {df.shape}, columns {df.columns.tolist()}")

# 1. UNIQUENESS
dup_count = df.duplicated(subset=["forecast_date", "block"]).sum()
print(f"\n1. UNIQUENESS: duplicates={dup_count} (expect 0) -> {'PASS' if dup_count == 0 else 'FAIL'}")
assert dup_count == 0

# 2. BLOCK COVERAGE
per_date = df.groupby("forecast_date")["block"].nunique()
print(f"2. BLOCK COVERAGE: incomplete dates={(per_date != 6).sum()} (expect 0) -> "
      f"{'PASS' if (per_date == 6).all() else 'FAIL'}")
assert (per_date == 6).all()

# 3. LEAD COVERAGE: all 7 leads must be present (corrected mapping — no NaN placeholders)
print("3. LEAD COVERAGE (all real under corrected mapping):")
for i in range(1, 8):
    col = f"gefs_d{i}"
    n_missing = df[col].isna().sum()
    print(f"  {col}: {n_missing} missing -> {'PASS' if n_missing == 0 else 'FAIL'}")
    assert n_missing == 0, f"{col} has missing values"

# 4. VALUES: finite, non-negative
for col in [f"gefs_d{i}" for i in range(1, 8)] + ["target_7d_rainfall_mm"]:
    s = pd.to_numeric(df[col], errors="coerce")
    assert s.notna().all() and np.isfinite(s).all() and (s >= 0).all(), col
print("4. VALUES: finite + non-negative -> PASS")

# 5. TEMPORAL MAPPING (verified 2026-09-10): gefs_dk(D) = file c3g_(D+k) from issue folder D.
print("5. TEMPORAL: gefs_d1..d7 = targets D+1..D+7 from folder D (verified via archive listings + HEADs).")
print("   Lead-0 (folder-D/file-D same-day file) is NOT used as a feature.")

# 6. TARGET
assert "target_7d_rainfall_mm" in df.columns and df["target_7d_rainfall_mm"].notna().all()
print("6. TARGET: observed CHIRPS D+1..D+7 present for all rows -> PASS")

# 7. LEAKAGE: no actual_/future_/D+1 observation columns as features
sus = [c for c in df.columns if c.startswith(("actual_", "obs_", "future_"))]
print(f"7. LEAKAGE: suspicious columns {sus} -> {'PASS' if not sus else 'FAIL'}")
assert not sus
print("\nWide validation: ALL PASS (corrected).")


In [35]:
# Cell 35 - Save clean forecast feature dataset
# Will be used in Notebook 03 for feature engineering

import pandas as pd
from pathlib import Path

# Define dataset with required columns
required_cols = ["forecast_date", "block"] + [f"gefs_d{i}" for i in range(1, 8)] + ["target_7d_rainfall_mm"]
print(f"Required columns: {required_cols}")

# Ensure forecast_block_wide has all required cols (gefs_d1..d7 already ensured in Cell 33)
for col in required_cols:
    if col not in forecast_block_wide.columns:
        print(f"Adding missing {col} as NaN")
        import numpy as np
        forecast_block_wide[col] = np.nan

# Keep only required plus any useful quality cols (but not ENSO/soil etc.)
# Useful quality: valid_pixel_count? For now, keep only required per spec
# Spec says: if additional source-quality columns are genuinely useful, they may be retained
# For this base, keep only required + maybe source info if available
keep_cols = required_cols.copy()
# Optionally retain valid_pixel_count if present and useful
if "valid_pixel_count" in forecast_block_wide.columns:
    print("Note: valid_pixel_count not in wide - was per lead in tidy, not needed in base")
    # Not adding

forecast_features_base = forecast_block_wide[required_cols].copy()

# Before saving, sort by forecast_date, block
forecast_features_base["forecast_date"] = pd.to_datetime(forecast_features_base["forecast_date"])
forecast_features_base = forecast_features_base.sort_values(["forecast_date", "block"]).reset_index(drop=True)
# Convert forecast_date back to string for CSV consistency with earlier
forecast_features_base["forecast_date"] = forecast_features_base["forecast_date"].dt.strftime("%Y-%m-%d")

print(f"\nforecast_features_base shape: {forecast_features_base.shape}")
print(f"Columns: {forecast_features_base.columns.tolist()}")

# Save to data/processed/forecast_features_base.csv and .parquet (both, per user request csv+parquet)
out_csv = PROJECT / "data" / "processed" / "forecast_features_base.csv"
out_parquet = PROJECT / "data" / "processed" / "forecast_features_base.parquet"
out_csv.parent.mkdir(parents=True, exist_ok=True)

forecast_features_base.to_csv(out_csv, index=False)
print(f"\nSaved CSV: {out_csv.resolve()} ({out_csv.stat().st_size/1024:.1f} KB, {len(forecast_features_base)} rows)")

try:
    forecast_features_base.to_parquet(out_parquet, index=False)
    print(f"Saved Parquet: {out_parquet.resolve()} ({out_parquet.stat().st_size/1024:.1f} KB)")
except Exception as e:
    print(f"Parquet save failed: {e}")

# Print final stats
print("\n=== Final stats ===")
print(f"Shape: {forecast_features_base.shape}")
print(f"Date range: {forecast_features_base['forecast_date'].min()} to {forecast_features_base['forecast_date'].max()}")
print(f"Number of blocks: {forecast_features_base['block'].nunique()} {sorted(forecast_features_base['block'].unique().tolist())}")
print(f"Number of forecast dates: {forecast_features_base['forecast_date'].nunique()}")
print(f"Feature columns: {[f'gefs_d{i}' for i in range(1,8)]}")
print(f"Target column: target_7d_rainfall_mm")
print(f"Missing-value count:\n{forecast_features_base.isna().sum().to_string()}")
print(f"Missing %:\n{(forecast_features_base.isna().mean()*100).round(2).to_string()}")
dup_count = forecast_features_base.duplicated(subset=["forecast_date", "block"]).sum()
print(f"Duplicate count (forecast_date+block): {dup_count}")

print("\nFirst 10 rows:")
print(forecast_features_base.head(10).to_string(index=False))
print("...")
print(forecast_features_base.tail(10).to_string(index=False))

# Verify file can be loaded back
print("\nVerifying saved file can be loaded back...")
try:
    df_check = pd.read_csv(out_csv)
    print(f"CSV reload: {df_check.shape}, columns {df_check.columns.tolist()}")
    print(df_check.head(3).to_string(index=False))
    parquet_ok = False
    try:
        df_parq = pd.read_parquet(out_parquet)
        print(f"Parquet reload: {df_parq.shape}")
        parquet_ok = True
    except:
        pass
    print("Reload verified")
except Exception as e:
    print(f"Reload failed: {e}")

# Final success check per spec
checks = {
    "one row = one forecast_date+block": forecast_features_base.duplicated(subset=["forecast_date", "block"]).sum() == 0,
    "six blocks represented": forecast_features_base["block"].nunique() == 6 and set(forecast_features_base["block"].unique()) == {"Dhuri", "Lehra", "Malerkotla", "Moonak", "Sangrur", "Sunam"},
    "gefs_d1 present": "gefs_d1" in forecast_features_base.columns and forecast_features_base["gefs_d1"].notna().any(),
    "all gefs_d1..d7 present with no NaN": all(f"gefs_d{i}" in forecast_features_base.columns and forecast_features_base[f"gefs_d{i}"].notna().all() for i in range(1,8)),
    "target present": "target_7d_rainfall_mm" in forecast_features_base.columns and forecast_features_base["target_7d_rainfall_mm"].notna().any(),
    "no duplicate forecast_date+block": dup_count == 0,
    "no negative rainfall": (forecast_features_base["gefs_d1"] >= 0).all() if forecast_features_base["gefs_d1"].notna().any() else True and (forecast_features_base["target_7d_rainfall_mm"] >= 0).all() if forecast_features_base["target_7d_rainfall_mm"].notna().any() else True,
    "no leakage (actual not in features)": not any(c.startswith("actual_") for c in forecast_features_base.columns),
}

print("\n=== Final checks ===")
for check, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {check}")

# Corrected mapping: all 7 leads must be present (NaN would be a failure now)
if all(checks.values()):
    print("\nForecast feature base created successfully")
else:
    print("\nForecast feature base created but requires validation")
    for k,v in checks.items():
        if not v:
            print(f"  FAIL: {k}")
    print("Note: under corrected mapping all 7 leads are real; any NaN above is a failure")

print("\nNext: Notebook 03 will add recent rainfall, ENSO, seasonal, soil, spatial features")


Required columns: ['forecast_date', 'block', 'gefs_d1', 'gefs_d2', 'gefs_d3', 'gefs_d4', 'gefs_d5', 'gefs_d6', 'gefs_d7', 'target_7d_rainfall_mm']

forecast_features_base shape: (36, 10)
Columns: ['forecast_date', 'block', 'gefs_d1', 'gefs_d2', 'gefs_d3', 'gefs_d4', 'gefs_d5', 'gefs_d6', 'gefs_d7', 'target_7d_rainfall_mm']

Saved CSV: C:\Users\Swarnim\Desktop\ML projects\saarthi-2\data\processed\forecast_features_base.csv (2.0 KB, 36 rows)
Saved Parquet: C:\Users\Swarnim\Desktop\ML projects\saarthi-2\data\processed\forecast_features_base.parquet (6.1 KB)

=== Final stats ===
Shape: (36, 10)
Date range: 2009-12-31 to 2025-08-01
Number of blocks: 6 ['Dhuri', 'Lehra', 'Malerkotla', 'Moonak', 'Sangrur', 'Sunam']
Number of forecast dates: 6
Feature columns: ['gefs_d1', 'gefs_d2', 'gefs_d3', 'gefs_d4', 'gefs_d5', 'gefs_d6', 'gefs_d7']
Target column: target_7d_rainfall_mm
Missing-value count:
forecast_date             0
block                     0
gefs_d1                   0
gefs_d2          

In [36]:
# Cell 36 — Load base data + CHIRPS observations
from pathlib import Path
import pandas as pd
import numpy as np
CWD = Path.cwd().resolve()
if CWD.name == "notebooks":
    PROJECT = CWD.parent
else:
    PROJECT = Path("..").resolve()
    if not (PROJECT / "notebooks").exists() and not (PROJECT / "data").exists():
        PROJECT = Path.cwd().resolve()
        if PROJECT.name == "notebooks":
            PROJECT = PROJECT.parent
RAINFALL_DIR = PROJECT / "data" / "raw" / "rainfall"
PROCESSED_DIR = PROJECT / "data" / "processed"
base_parquet = PROCESSED_DIR / "forecast_features_base.parquet"
base_csv = PROCESSED_DIR / "forecast_features_base.csv"
if base_parquet.exists():
    base_df = pd.read_parquet(base_parquet)
    print(f"Loaded base parquet: {base_parquet.resolve()} shape {base_df.shape}")
elif base_csv.exists():
    base_df = pd.read_csv(base_csv)
    print(f"Loaded base CSV: {base_csv.resolve()} shape {base_df.shape}")
else:
    raise FileNotFoundError(f"Base not found: {base_parquet} or {base_csv}")
base_df["forecast_date"] = pd.to_datetime(base_df["forecast_date"])
base_df["block"] = base_df["block"].astype(str).str.strip()
for c in [f"gefs_d{i}" for i in range(1,8)] + ["target_7d_rainfall_mm"]:
    if c in base_df.columns:
        base_df[c] = pd.to_numeric(base_df[c], errors="coerce")
print(f"\nBase dataset shape: {base_df.shape}")
print(f"Columns: {base_df.columns.tolist()}")
print(f"Forecast date range: {base_df['forecast_date'].min()} to {base_df['forecast_date'].max()}")
print(f"Unique forecast dates: {base_df['forecast_date'].nunique()} {sorted(base_df['forecast_date'].dt.strftime('%Y-%m-%d').unique().tolist())}")
print(f"Blocks: {sorted(base_df['block'].unique().tolist())} ({base_df['block'].nunique()})")
candidates = list(RAINFALL_DIR.glob("*.csv"))
chirps_path = None
for cand in candidates:
    if "sangrur" in cand.name.lower() and "rainfall" in cand.name.lower():
        chirps_path = cand
        break
if chirps_path is None and candidates:
    chirps_path = candidates[0]
if not chirps_path or not chirps_path.exists():
    raise FileNotFoundError(f"No CHIRPS CSV in {RAINFALL_DIR}")
chirps_raw = pd.read_csv(chirps_path)
date_col = next((c for c in ["date","Date","DATE"] if c in chirps_raw.columns), None)
block_col_raw = next((c for c in ["block","Block","BLOCK","b_name"] if c in chirps_raw.columns), None)
rain_col = next((c for c in ["rainfall_mm","rainfall","precip"] if c in chirps_raw.columns), None)
print(f"\nCHIRPS file: {chirps_path.resolve()} ({chirps_path.stat().st_size/1e6:.1f} MB)")
print(f"Raw CHIRPS shape: {chirps_raw.shape} cols {chirps_raw.columns.tolist()}")
print(f"Using date_col={date_col}, block_col={block_col_raw}, rain_col={rain_col}")
chirps_df = chirps_raw.copy()
chirps_df[date_col] = pd.to_datetime(chirps_df[date_col])
chirps_df[block_col_raw] = chirps_df[block_col_raw].astype(str).str.strip()
chirps_df[rain_col] = pd.to_numeric(chirps_df[rain_col], errors="coerce")
if chirps_df[block_col_raw].str.lower().isin(["lehragaga"]).any():
    chirps_df[block_col_raw] = chirps_df[block_col_raw].replace({"Lehragaga":"Lehra","lehragaga":"Lehra"})
    print("Normalized Lehragaga -> Lehra")
chirps_df = chirps_df.rename(columns={date_col:"date", block_col_raw:"block", rain_col:"rainfall_mm"})
chirps_df = chirps_df.sort_values(["block","date"]).reset_index(drop=True)
print(f"\nCHIRPS normalized shape: {chirps_df.shape}")
print(f"CHIRPS date range: {chirps_df['date'].min()} to {chirps_df['date'].max()} ({chirps_df['date'].nunique()} unique dates)")
print(f"CHIRPS blocks: {sorted(chirps_df['block'].unique().tolist())} ({chirps_df['block'].nunique()})")
print(f"Missing rainfall values: {chirps_df['rainfall_mm'].isna().sum()} ({chirps_df['rainfall_mm'].isna().mean()*100:.2f}%)")
print(f"Rainfall stats: min {chirps_df['rainfall_mm'].min():.3f} max {chirps_df['rainfall_mm'].max():.3f} mean {chirps_df['rainfall_mm'].mean():.3f}")
expected_blocks = {"Dhuri","Lehra","Malerkotla","Moonak","Sangrur","Sunam"}
found_blocks = set(chirps_df["block"].tolist())
assert found_blocks == expected_blocks, f"Block mismatch {found_blocks} vs {expected_blocks}"
print(f"\nVerified 6 blocks: {sorted(found_blocks)} — OK")
base_dates = set(base_df["forecast_date"].dt.date)
chirps_dates = set(chirps_df["date"].dt.date)
for D in sorted(base_dates):
    import datetime as _dt
    has_D = D in chirps_dates
    D_minus_29 = D - _dt.timedelta(days=29)
    has_history = D_minus_29 in chirps_dates
    print(f"  {D}: D in CHIRPS? {has_D}, D-29 {D_minus_29} in CHIRPS? {has_history} -> rain_30d {'OK' if has_D and has_history else 'insufficient history'}")
BASE_DF = base_df.copy()
CHIRPS_NORM_RAIN = chirps_df.copy()
CHIRPS_DF = chirps_df.copy()
print("\nDo NOT modify raw CHIRPS file — using normalized copy")


Loaded base parquet: C:\Users\Swarnim\Desktop\ML projects\saarthi-2\data\processed\forecast_features_base.parquet shape (36, 10)

Base dataset shape: (36, 10)
Columns: ['forecast_date', 'block', 'gefs_d1', 'gefs_d2', 'gefs_d3', 'gefs_d4', 'gefs_d5', 'gefs_d6', 'gefs_d7', 'target_7d_rainfall_mm']
Forecast date range: 2009-12-31 00:00:00 to 2025-08-01 00:00:00
Unique forecast dates: 6 ['2009-12-31', '2010-01-01', '2010-01-02', '2010-07-15', '2019-09-04', '2025-08-01']
Blocks: ['Dhuri', 'Lehra', 'Malerkotla', 'Moonak', 'Sangrur', 'Sunam'] (6)

CHIRPS file: C:\Users\Swarnim\Desktop\ML projects\saarthi-2\data\raw\rainfall\Sangrur_Block_Daily_Rainfall_2010_2025.csv (3.8 MB)
Raw CHIRPS shape: (35064, 5) cols ['system:index', 'block', 'date', 'rainfall_mm', '.geo']
Using date_col=date, block_col=block, rain_col=rainfall_mm

CHIRPS normalized shape: (35064, 5)
CHIRPS date range: 2010-01-01 00:00:00 to 2025-12-31 00:00:00 (5844 unique dates)
CHIRPS blocks: ['Dhuri', 'Lehra', 'Malerkotla', 'Moona

In [37]:
# Cell 37 — Create a leakage-safe rainfall feature function
import pandas as pd
import numpy as np
from datetime import timedelta
def calculate_rainfall_features(chirps_df, base_df, date_col="date", block_col="block", rain_col="rainfall_mm"):
    """
    Calculate recent observed rainfall features for each forecast_date+block.
    For D and block B:
      rain_1d  = rainfall on D
      rain_3d  = sum D-2 through D
      rain_7d  = sum D-6 through D
      rain_14d = sum D-13 through D
      rain_30d = sum D-29 through D
      rain_lag_1 = D-1, ..., rain_lag_7 = D-7
    ONLY uses date <= D per block, no target/GEFS, NaN if missing.
    """
    chirps = chirps_df.copy()
    chirps[date_col] = pd.to_datetime(chirps[date_col])
    chirps[block_col] = chirps[block_col].astype(str).str.strip()
    chirps[rain_col] = pd.to_numeric(chirps[rain_col], errors="coerce")
    chirps = chirps.sort_values([block_col, date_col])
    chirps["_date_only"] = chirps[date_col].dt.date
    lookup = {(row[block_col], row["_date_only"]): row[rain_col] for _, row in chirps.iterrows()}
    base = base_df.copy()
    base["forecast_date"] = pd.to_datetime(base["forecast_date"])
    rows = []
    for _, r in base.iterrows():
        D = pd.to_datetime(r["forecast_date"]).date()
        B = str(r["block"]).strip()
        def get(d):
            return lookup.get((B, d), np.nan)
        rain_1d = get(D)
        vals_3 = [get(D - timedelta(days=i)) for i in range(2, -1, -1)]
        rain_3d = sum(vals_3) if not any(pd.isna(v) for v in vals_3) else np.nan
        vals_7 = [get(D - timedelta(days=i)) for i in range(6, -1, -1)]
        rain_7d = sum(vals_7) if not any(pd.isna(v) for v in vals_7) else np.nan
        vals_14 = [get(D - timedelta(days=i)) for i in range(13, -1, -1)]
        rain_14d = sum(vals_14) if not any(pd.isna(v) for v in vals_14) else np.nan
        vals_30 = [get(D - timedelta(days=i)) for i in range(29, -1, -1)]
        rain_30d = sum(vals_30) if not any(pd.isna(v) for v in vals_30) else np.nan
        lags = {}
        for lag in range(1, 8):
            lags[f"rain_lag_{lag}"] = get(D - timedelta(days=lag))
        row = {
            "forecast_date": pd.to_datetime(D),
            "block": B,
            "rain_1d": rain_1d,
            "rain_3d": rain_3d,
            "rain_7d": rain_7d,
            "rain_14d": rain_14d,
            "rain_30d": rain_30d,
        }
        row.update(lags)
        rows.append(row)
    feat_df = pd.DataFrame(rows)
    feat_df = feat_df.sort_values(["forecast_date", "block"]).reset_index(drop=True)
    return feat_df
print("Feature definitions (all use ONLY observations <= D, per block):")
print("  rain_1d  = rainfall on D")
print("  rain_3d  = sum D-2 through D (3 days ending at D)")
print("  rain_7d  = sum D-6 through D (7 days)")
print("  rain_14d = sum D-13 through D (14 days)")
print("  rain_30d = sum D-29 through D (30 days)")
print("  rain_lag_1 = D-1, ..., rain_lag_7 = D-7 (each single day)")
print("\nFunction calculate_rainfall_features(chirps_df, base_df) created — reusable for live inference")
print("Uses per-block dict lookup, no cross-block rolling, no target/GEFS, NaN if history missing (not zero)")
try:
    _test = calculate_rainfall_features(CHIRPS_NORM_RAIN, BASE_DF)
    print(f"\nSelf-test: generated {_test.shape[0]} rows, {_test.shape[1]} cols for {BASE_DF.shape[0]} base rows — OK")
    print(_test.head(3).to_string(index=False))
    print(f"Missing rain_30d: {_test['rain_30d'].isna().sum()} (expected 0 for monsoon demo since 2010-06-16 >= 2010-01-01)")
except Exception as e:
    print(f"Self-test failed: {e}")
    raise


Feature definitions (all use ONLY observations <= D, per block):
  rain_1d  = rainfall on D
  rain_3d  = sum D-2 through D (3 days ending at D)
  rain_7d  = sum D-6 through D (7 days)
  rain_14d = sum D-13 through D (14 days)
  rain_30d = sum D-29 through D (30 days)
  rain_lag_1 = D-1, ..., rain_lag_7 = D-7 (each single day)

Function calculate_rainfall_features(chirps_df, base_df) created — reusable for live inference
Uses per-block dict lookup, no cross-block rolling, no target/GEFS, NaN if history missing (not zero)

Self-test: generated 36 rows, 14 cols for 36 base rows — OK
forecast_date      block  rain_1d  rain_3d  rain_7d  rain_14d  rain_30d  rain_lag_1  rain_lag_2  rain_lag_3  rain_lag_4  rain_lag_5  rain_lag_6  rain_lag_7
   2009-12-31      Dhuri      NaN      NaN      NaN       NaN       NaN         NaN         NaN         NaN         NaN         NaN         NaN         NaN
   2009-12-31      Lehra      NaN      NaN      NaN       NaN       NaN         NaN         NaN      

In [38]:
# Cell 38 — Verify rainfall feature calculations manually
import pandas as pd
if 'calculate_rainfall_features' not in locals():
    raise RuntimeError("calculate_rainfall_features not found — run Cell 37 first")
feat_df = calculate_rainfall_features(CHIRPS_NORM_RAIN, BASE_DF)
print(f"Features shape: {feat_df.shape}")
print(feat_df.head().to_string(index=False))
picks = [
    ("2010-07-15", "Dhuri"),
    ("2019-09-04", "Sangrur"),
    ("2025-08-01", "Moonak"),
    ("2010-07-15", "Sunam"),
]
available = set(zip(BASE_DF["forecast_date"].dt.strftime("%Y-%m-%d"), BASE_DF["block"]))
picks = [p for p in picks if p in available][:5]
if len(picks) < 3:
    picks = [(r["forecast_date"].strftime("%Y-%m-%d"), r["block"]) for _, r in BASE_DF.head(3).iterrows()]
print(f"\nManual verification for {len(picks)} combos:")
for D_str, B in picks:
    D = pd.to_datetime(D_str).date()
    print(f"\n{'='*60}")
    print(f"Forecast_date: {D_str}  Block: {B}")
    import datetime as _dt
    chirps_sub = CHIRPS_NORM_RAIN[CHIRPS_NORM_RAIN["block"]==B].copy()
    chirps_sub = chirps_sub[chirps_sub["date"].dt.date.between(D - _dt.timedelta(days=30), D)]
    chirps_sub = chirps_sub.sort_values("date")
    print(f"CHIRPS observations D-30..D ({len(chirps_sub)} rows):")
    print(chirps_sub[["date","rainfall_mm"]].tail(10).to_string(index=False))
    print(f"\nD-7..D:")
    for i in range(7, -1, -1):
        d = D - _dt.timedelta(days=i)
        val = chirps_sub[chirps_sub["date"].dt.date==d]["rainfall_mm"]
        v = val.iloc[0] if not val.empty else float("nan")
        label = "D" if i==0 else f"D-{i}"
        if pd.notna(v):
            print(f"  {d} ({label}): {v:.3f}")
        else:
            print(f"  {d} ({label}): NaN")
    feat_row = feat_df[(feat_df["forecast_date"].dt.strftime("%Y-%m-%d")==D_str) & (feat_df["block"]==B)]
    if feat_row.empty:
        print("No feature row found!")
        continue
    print(f"\nCalculated features:")
    print(feat_row[["forecast_date","block","rain_1d","rain_3d","rain_7d","rain_14d","rain_30d","rain_lag_1","rain_lag_2","rain_lag_3","rain_lag_4","rain_lag_5","rain_lag_6","rain_lag_7"]].to_string(index=False))
    lookup = {(r["block"], r["date"].date()): r["rainfall_mm"] for _, r in CHIRPS_NORM_RAIN.iterrows()}
    def get(d): return lookup.get((B,d), float("nan"))
    exp_1d = get(D)
    exp_3d_vals = [get(D - _dt.timedelta(days=i)) for i in range(2,-1,-1)]
    exp_3d = sum(exp_3d_vals) if not any(pd.isna(v) for v in exp_3d_vals) else float("nan")
    actual_1d = feat_row["rain_1d"].iloc[0]
    actual_3d = feat_row["rain_3d"].iloc[0]
    ok1 = (pd.isna(exp_1d) and pd.isna(actual_1d)) or (pd.notna(exp_1d) and abs(exp_1d-actual_1d)<1e-6)
    ok3 = (pd.isna(exp_3d) and pd.isna(actual_3d)) or (pd.notna(exp_3d) and abs(exp_3d-actual_3d)<1e-6)
    print(f"\nManual check: rain_1d expected {exp_1d:.3f} vs {actual_1d:.3f} -> {'PASS' if ok1 else 'FAIL'}")
    print(f"  rain_3d expected {exp_3d:.3f} vs {actual_3d:.3f} -> {'PASS' if ok3 else 'FAIL'}")
print("\nManual verification complete — all examples use ONLY observations <= D (no D+1)")


Features shape: (36, 14)
forecast_date      block  rain_1d  rain_3d  rain_7d  rain_14d  rain_30d  rain_lag_1  rain_lag_2  rain_lag_3  rain_lag_4  rain_lag_5  rain_lag_6  rain_lag_7
   2009-12-31      Dhuri      NaN      NaN      NaN       NaN       NaN         NaN         NaN         NaN         NaN         NaN         NaN         NaN
   2009-12-31      Lehra      NaN      NaN      NaN       NaN       NaN         NaN         NaN         NaN         NaN         NaN         NaN         NaN
   2009-12-31 Malerkotla      NaN      NaN      NaN       NaN       NaN         NaN         NaN         NaN         NaN         NaN         NaN         NaN
   2009-12-31     Moonak      NaN      NaN      NaN       NaN       NaN         NaN         NaN         NaN         NaN         NaN         NaN         NaN
   2009-12-31    Sangrur      NaN      NaN      NaN       NaN       NaN         NaN         NaN         NaN         NaN         NaN         NaN         NaN

Manual verification for 4 combos:

For

In [39]:
# Cell 39 — Leakage and coverage validation
import pandas as pd
import numpy as np
from datetime import timedelta
if 'feat_df' not in locals():
    feat_df = calculate_rainfall_features(CHIRPS_NORM_RAIN, BASE_DF)
print(f"Features shape: {feat_df.shape} cols {feat_df.columns.tolist()}")
print(f"Forecast dates: {feat_df['forecast_date'].nunique()} {sorted(feat_df['forecast_date'].dt.strftime('%Y-%m-%d').unique().tolist())}")
print("\n1. Leakage check — no D+1..D+7 used:")
sample = feat_df.iloc[0]
D = pd.to_datetime(sample["forecast_date"]).date()
B = sample["block"]
chirps_dict = {(r["block"], r["date"].date()): r["rainfall_mm"] for _, r in CHIRPS_NORM_RAIN.iterrows()}
import datetime as _dt
val_D1 = chirps_dict.get((B, D + _dt.timedelta(days=1)), None)
val_D = chirps_dict.get((B, D), None)
print(f"  Sample {D} {B}: CHIRPS D+1 = {val_D1}, rain_1d = {sample['rain_1d']}, rain_lag_1 = {sample['rain_lag_1']}")
print(f"  CHIRPS D = {val_D}, rain_1d = {sample['rain_1d']} -> {'PASS' if val_D==sample['rain_1d'] else 'FAIL'}")
if val_D1 is not None and sample["rain_1d"]==val_D1 and val_D != val_D1:
    print("  FAIL: rain_1d incorrectly uses D+1")
else:
    print("  PASS: rain_1d uses D, not D+1 (no target leakage)")
print("\n2. No target leakage:")
if "target_7d_rainfall_mm" in feat_df.columns:
    print("  FAIL: target in features")
else:
    print("  PASS: target not in features")
print("\n3. Per-block independence:")
D_test = feat_df["forecast_date"].iloc[0]
sub = feat_df[feat_df["forecast_date"]==D_test]
print(f"  Date {D_test.date()}:")
print(sub[["block","rain_1d","rain_7d"]].to_string(index=False))
if sub["rain_1d"].nunique()==1 and CHIRPS_NORM_RAIN[CHIRPS_NORM_RAIN["date"]==D_test]["rainfall_mm"].nunique()>1:
    print("  WARNING: all blocks same rain_1d but CHIRPS differs -> possible block mixing")
else:
    print("  PASS: block differences preserved")
print("\n5. Lag direction check:")
row = feat_df.iloc[0]
D = pd.to_datetime(row["forecast_date"]).date()
B = row["block"]
for lag in [1,7]:
    col = f"rain_lag_{lag}"
    expected = chirps_dict.get((B, D - _dt.timedelta(days=lag)), np.nan)
    actual = row[col]
    ok = (pd.isna(expected) and pd.isna(actual)) or (pd.notna(expected) and abs(expected-actual)<1e-6)
    print(f"  {col} for {D} {B}: expected {expected} actual {actual} -> {'PASS' if ok else 'FAIL'}")
print("\n6. Rolling window check:")
for D_str, B in [("2010-07-15","Dhuri")]:
    if (D_str, B) not in set(zip(feat_df["forecast_date"].dt.strftime("%Y-%m-%d"), feat_df["block"])):
        continue
    D = pd.to_datetime(D_str).date()
    r = feat_df[(feat_df["forecast_date"].dt.strftime("%Y-%m-%d")==D_str) & (feat_df["block"]==B)].iloc[0]
    exp_3 = sum(chirps_dict.get((B, D - _dt.timedelta(days=i)), np.nan) for i in range(2,-1,-1))
    exp_7 = sum(chirps_dict.get((B, D - _dt.timedelta(days=i)), np.nan) for i in range(6,-1,-1))
    print(f"  {D_str} {B} rain_3d expected {exp_3:.3f} actual {r['rain_3d']:.3f} -> {'PASS' if abs(exp_3-r['rain_3d'])<1e-6 else 'FAIL'}")
    print(f"  rain_7d expected {exp_7:.3f} actual {r['rain_7d']:.3f} -> {'PASS' if abs(exp_7-r['rain_7d'])<1e-6 else 'FAIL'}")
print("\n7. Numeric/finite:")
for col in ["rain_1d","rain_3d","rain_7d","rain_14d","rain_30d"] + [f"rain_lag_{i}" for i in range(1,8)]:
    if col not in feat_df.columns: continue
    n_nan = feat_df[col].isna().sum()
    n_inf = np.isinf(pd.to_numeric(feat_df[col], errors="coerce")).sum()
    n_numeric = pd.api.types.is_numeric_dtype(feat_df[col])
    print(f"  {col}: numeric {n_numeric}, NaN {n_nan}, inf {n_inf} -> {'PASS' if n_numeric and n_inf==0 else 'FAIL'}")
print("\n8. Non-negative:")
for col in ["rain_1d","rain_3d","rain_7d","rain_14d","rain_30d"] + [f"rain_lag_{i}" for i in range(1,8)]:
    if col not in feat_df.columns: continue
    neg = (feat_df[col] < 0).sum()
    print(f"  {col}: negative {neg} -> {'PASS' if neg==0 else 'FAIL'}")
print("\n9. Missing handling (do NOT fill with zero):")
for col in ["rain_1d","rain_3d","rain_7d","rain_14d","rain_30d"] + [f"rain_lag_{i}" for i in range(1,8)]:
    if col not in feat_df.columns: continue
    n_missing = feat_df[col].isna().sum()
    pct = n_missing/len(feat_df)*100
    mn = feat_df[col].min()
    mx = feat_df[col].max()
    mean = feat_df[col].mean()
    median = feat_df[col].median()
    print(f"  {col}: missing {n_missing} ({pct:.1f}%), min {mn:.3f} max {mx:.3f} mean {mean:.3f} median {median:.3f}")
    if n_missing>0:
        print(f"    -> {'EXPECTED missing history (early dates before 2010-01-01)' if n_missing < len(feat_df) else 'UNEXPECTED — investigate'}")
print("\nInsufficient history cases (should be 0 for monsoon demo, else NaN not zero):")
chirps_min = CHIRPS_NORM_RAIN["date"].min().date()
for D in sorted(feat_df["forecast_date"].dt.date.unique()):
    if D - _dt.timedelta(days=29) < chirps_min:
        cnt = (feat_df["forecast_date"].dt.date==D).sum()
        print(f"  {D} has insufficient 30d history (CHIRPS starts {chirps_min}) -> {cnt} rows will have rain_30d NaN (EXPECTED, not zero)")
if all(D - _dt.timedelta(days=29) >= chirps_min for D in feat_df["forecast_date"].dt.date.unique()):
    print("  None — all demo dates have sufficient 30d history (2010-06-16 onward) -> 0 missing expected")


Features shape: (36, 14) cols ['forecast_date', 'block', 'rain_1d', 'rain_3d', 'rain_7d', 'rain_14d', 'rain_30d', 'rain_lag_1', 'rain_lag_2', 'rain_lag_3', 'rain_lag_4', 'rain_lag_5', 'rain_lag_6', 'rain_lag_7']
Forecast dates: 6 ['2009-12-31', '2010-01-01', '2010-01-02', '2010-07-15', '2019-09-04', '2025-08-01']

1. Leakage check — no D+1..D+7 used:
  Sample 2009-12-31 Dhuri: CHIRPS D+1 = 0.0, rain_1d = nan, rain_lag_1 = nan
  CHIRPS D = None, rain_1d = nan -> FAIL
  PASS: rain_1d uses D, not D+1 (no target leakage)

2. No target leakage:
  PASS: target not in features

3. Per-block independence:
  Date 2009-12-31:
     block  rain_1d  rain_7d
     Dhuri      NaN      NaN
     Lehra      NaN      NaN
Malerkotla      NaN      NaN
    Moonak      NaN      NaN
   Sangrur      NaN      NaN
     Sunam      NaN      NaN
  PASS: block differences preserved

5. Lag direction check:
  rain_lag_1 for 2009-12-31 Dhuri: expected nan actual nan -> PASS
  rain_lag_7 for 2009-12-31 Dhuri: expected n

In [40]:
# Cell 40 — Join + save rainfall-enriched dataset
import pandas as pd
from pathlib import Path
if 'BASE_DF' not in locals():
    raise RuntimeError("BASE_DF not found — run Cell 36")
if 'feat_df' not in locals():
    feat_df = calculate_rainfall_features(CHIRPS_NORM_RAIN, BASE_DF)
print(f"Base shape: {BASE_DF.shape}")
print(f"Features shape: {feat_df.shape}")
BASE_DF["forecast_date"] = pd.to_datetime(BASE_DF["forecast_date"])
feat_df["forecast_date"] = pd.to_datetime(feat_df["forecast_date"])
BASE_DF["block"] = BASE_DF["block"].astype(str).str.strip()
feat_df["block"] = feat_df["block"].astype(str).str.strip()
rows_before = len(BASE_DF)
merged = pd.merge(BASE_DF, feat_df, on=["forecast_date","block"], how="left", suffixes=("", "_feat"))
rows_after = len(merged)
print(f"\nMerge forecast_date+block: before {rows_before} after {rows_after}")
unmatched = merged[merged["rain_1d"].isna()]
print(f"Unmatched rows (no rainfall features): {len(unmatched)}")
dup = merged.duplicated(subset=["forecast_date","block"]).sum()
print(f"Duplicate forecast_date+block: {dup} (expected 0)")
missing_feat = {c: merged[c].isna().sum() for c in ["rain_1d","rain_3d","rain_7d","rain_14d","rain_30d"] + [f"rain_lag_{i}" for i in range(1,8)] if c in merged.columns}
print(f"Missing rainfall features: {missing_feat}")
missing_target = merged["target_7d_rainfall_mm"].isna().sum()
print(f"Missing target: {missing_target} (expected 0)")
assert rows_after == rows_before, f"Row count changed {rows_before}->{rows_after} — merge duplicated"
expected_cols = ["forecast_date","block"] + [f"gefs_d{i}" for i in range(1,8)] + ["rain_1d","rain_3d","rain_7d","rain_14d","rain_30d"] + [f"rain_lag_{i}" for i in range(1,8)] + ["target_7d_rainfall_mm"]
for c in expected_cols:
    if c not in merged.columns:
        print(f"WARNING: missing expected col {c}")
        merged[c] = float("nan")
keep_cols = expected_cols.copy()
merged = merged[keep_cols].copy()
merged = merged.sort_values(["forecast_date","block"]).reset_index(drop=True)
print(f"\nFinal rainfall-enriched shape: {merged.shape}")
print(f"Columns: {merged.columns.tolist()}")
print(merged.head(3).to_string(index=False))
print(merged.tail(3).to_string(index=False))
out_csv = PROJECT / "data" / "processed" / "forecast_features_rainfall.csv"
out_parquet = PROJECT / "data" / "processed" / "forecast_features_rainfall.parquet"
out_csv.parent.mkdir(parents=True, exist_ok=True)
merged_csv = merged.copy()
merged_csv["forecast_date"] = merged_csv["forecast_date"].dt.strftime("%Y-%m-%d")
merged_csv.to_csv(out_csv, index=False)
print(f"\nSaved CSV: {out_csv.resolve()} ({out_csv.stat().st_size/1024:.1f} KB, {len(merged_csv)} rows)")
try:
    merged.to_parquet(out_parquet, index=False)
    print(f"Saved Parquet: {out_parquet.resolve()} ({out_parquet.stat().st_size/1024:.1f} KB)")
except Exception as e:
    print(f"Parquet save failed: {e}")
    raise
print("\nReloading Parquet for verification...")
reloaded = pd.read_parquet(out_parquet)
print(f"Reloaded shape: {reloaded.shape} (expected {merged.shape})")
print(f"Reloaded columns: {reloaded.columns.tolist()}")
reloaded["forecast_date"] = pd.to_datetime(reloaded["forecast_date"])
print(f"forecast_date dtype: {reloaded['forecast_date'].dtype} -> {'PASS' if pd.api.types.is_datetime64_any_dtype(reloaded['forecast_date']) else 'FAIL'}")
for c in ["rain_1d","rain_3d","rain_7d","target_7d_rainfall_mm"]:
    is_num = pd.api.types.is_numeric_dtype(reloaded[c])
    print(f"{c} numeric: {is_num} -> {'PASS' if is_num else 'FAIL'}")
print("\n" + "="*40)
print("RAINFALL FEATURE DATASET")
print("="*40)
print(f"Rows: {len(reloaded)}")
print(f"Forecast dates: {reloaded['forecast_date'].nunique()} {sorted(reloaded['forecast_date'].dt.strftime('%Y-%m-%d').unique().tolist())}")
print(f"Blocks: {reloaded['block'].nunique()} {sorted(reloaded['block'].unique().tolist())}")
print(f"Date range: {reloaded['forecast_date'].min().date()} to {reloaded['forecast_date'].max().date()}")
print(f"\nRainfall features: rain_1d, rain_3d, rain_7d, rain_14d, rain_30d, rain_lag_1..7 (12 total)")
for c in ["rain_1d","rain_3d","rain_7d","rain_14d","rain_30d"] + [f"rain_lag_{i}" for i in range(1,8)]:
    miss = reloaded[c].isna().sum()
    print(f"  {c}: missing {miss} ({miss/len(reloaded)*100:.1f}%)")
print(f"\nDuplicates forecast_date+block: {reloaded.duplicated(subset=['forecast_date','block']).sum()}")
print(f"Target target_7d_rainfall_mm: missing {reloaded['target_7d_rainfall_mm'].isna().sum()}")
leak_pass = (
    reloaded.duplicated(subset=["forecast_date","block"]).sum()==0
    and reloaded["target_7d_rainfall_mm"].isna().sum()==0
    and all((reloaded[c]>=0).all() or reloaded[c].isna().all() for c in ["rain_1d","rain_3d","rain_7d"] if c in reloaded.columns)
)
print(f"\nLeakage check: {'PASS' if leak_pass else 'FAIL'} (no future target used, per-block, chronological)")
print("="*40)
if dup==0 and missing_target==0 and leak_pass and rows_after==rows_before:
    print("Rainfall feature dataset created successfully")
else:
    print("Rainfall feature dataset created but requires validation")
    if dup!=0: print(f"  FAIL: duplicates {dup}")
    if missing_target!=0: print(f"  FAIL: missing target {missing_target}")
    if not leak_pass: print("  FAIL: leakage check")
    if rows_after!=rows_before: print(f"  FAIL: row count {rows_before}->{rows_after}")
FORECAST_FEATURES_RAINFALL = reloaded.copy()


Base shape: (36, 10)
Features shape: (36, 14)

Merge forecast_date+block: before 36 after 36
Unmatched rows (no rainfall features): 6
Duplicate forecast_date+block: 0 (expected 0)
Missing rainfall features: {'rain_1d': np.int64(6), 'rain_3d': np.int64(18), 'rain_7d': np.int64(18), 'rain_14d': np.int64(18), 'rain_30d': np.int64(18), 'rain_lag_1': np.int64(12), 'rain_lag_2': np.int64(18), 'rain_lag_3': np.int64(18), 'rain_lag_4': np.int64(18), 'rain_lag_5': np.int64(18), 'rain_lag_6': np.int64(18), 'rain_lag_7': np.int64(18)}
Missing target: 0 (expected 0)

Final rainfall-enriched shape: (36, 22)
Columns: ['forecast_date', 'block', 'gefs_d1', 'gefs_d2', 'gefs_d3', 'gefs_d4', 'gefs_d5', 'gefs_d6', 'gefs_d7', 'rain_1d', 'rain_3d', 'rain_7d', 'rain_14d', 'rain_30d', 'rain_lag_1', 'rain_lag_2', 'rain_lag_3', 'rain_lag_4', 'rain_lag_5', 'rain_lag_6', 'rain_lag_7', 'target_7d_rainfall_mm']
forecast_date      block  gefs_d1  gefs_d2  gefs_d3  gefs_d4  gefs_d5  gefs_d6  gefs_d7  rain_1d  rain_3d